In [ ]:
!pip install -q -U langchain-groq
!pip install -q pandas==2.2.3

In [ ]:
import pandas as pd
import langchain_groq

print("Pandas version:", pd.__version__)
print("LangChain Groq installed successfully!")

Pandas version: 2.2.3
LangChain Groq installed successfully!


In [ ]:
import os
from getpass import getpass

groq_api_key = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API key loaded successfully!")

Enter your Groq API key: ··········
Groq API key loaded successfully!


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [ ]:
response = llm.invoke(
    "Explain in one sentence what a data analyst does."
)

print(response.content)

A data analyst collects, cleans, and examines data to uncover patterns, generate insights, and support decision‑making through visualizations and reports.


In [ ]:
job_description = """
A Data Analyst collects, cleans, processes, and analyzes data to identify trends
and generate insights. The role involves creating dashboards and reports,
performing statistical analysis, validating data quality, working with
stakeholders to understand business requirements, and presenting findings
to support business decision-making.
"""

print(job_description)


A Data Analyst collects, cleans, processes, and analyzes data to identify trends
and generate insights. The role involves creating dashboards and reports,
performing statistical analysis, validating data quality, working with
stakeholders to understand business requirements, and presenting findings
to support business decision-making.



In [ ]:
import json

task_prompt = """
You are a job-task analysis assistant.

Read the job description below and extract the major tasks performed in this role.

Rules:
- Extract between 6 and 10 major tasks.
- Each task must describe a concrete work activity.
- Keep each task concise.
- Do not include personality traits.
- Do not include generic statements.
- Do not analyze AI impact.
- Do not assign scores.
- Return ONLY valid JSON.
- Do not use Markdown or code fences.

Return exactly this structure:
[
  {"task": "First concrete task"},
  {"task": "Second concrete task"}
]

Job description:
""" + job_description

response = llm.invoke(task_prompt)

content = response.content

if isinstance(content, list):
    content = "".join(
        item.get("text", "") if isinstance(item, dict) else str(item)
        for item in content
    )

content = content.strip()

# Remove Markdown code fences if the model adds them
if content.startswith("```"):
    content = content.replace("```json", "").replace("```", "").strip()

# Locate the JSON array
start = content.find("[")
end = content.rfind("]")

if start == -1 or end == -1:
    raise ValueError(
        "The model did not return a valid JSON array.\n\n"
        + content
    )

content = content[start:end + 1]

tasks = json.loads(content)

# Validate the task structure
if not isinstance(tasks, list):
    raise ValueError("Tasks must be returned as a list.")

if not 6 <= len(tasks) <= 10:
    raise ValueError(
        f"Expected 6–10 tasks, but received {len(tasks)}."
    )

for i, item in enumerate(tasks):
    if not isinstance(item, dict) or "task" not in item:
        raise ValueError(
            f"Invalid task structure at position {i}: {item}"
        )

print("Task extraction successful!")
print("Number of tasks:", len(tasks))

for i, item in enumerate(tasks):
    print(f"{i + 1}. {item['task']}")

Task extraction successful!
Number of tasks: 7
1. Collect raw data from various internal and external sources
2. Clean and preprocess data to ensure accuracy and consistency
3. Perform statistical analysis to identify trends and patterns
4. Develop and maintain interactive dashboards and reports
5. Validate data quality and resolve inconsistencies
6. Collaborate with stakeholders to gather and clarify business requirements
7. Present analytical findings and insights to support decision-making


In [ ]:
import pandas as pd

df_tasks = pd.DataFrame(tasks)

print("Number of tasks:", len(df_tasks))
display(df_tasks)

Number of tasks: 7


,task
0,Collect raw data from various internal and ext...
1,Clean and preprocess data to ensure accuracy a...
2,Perform statistical analysis to identify trend...
3,Develop and maintain interactive dashboards an...
4,Validate data quality and resolve inconsistencies
5,Collaborate with stakeholders to gather and cl...
6,Present analytical findings and insights to su...


In [ ]:
import json

tasks_text = "\n".join(
    f"{i + 1}. {task}"
    for i, task in enumerate(df_tasks["task"])
)

impact_prompt = """
You are an expert in AI and workplace transformation.

Analyze the following job tasks and evaluate how AI may affect each task.

For every task, provide:

1. automation_potential:
- Low
- Medium
- High

2. human_dependency:
- Low
- Medium
- High

3. ai_suitability:
- Low
- Medium
- High

Definitions:

automation_potential:
How much of the task's work could potentially be automated using AI or software.

human_dependency:
How strongly the task depends on human judgment, communication, collaboration,
context, or decision-making.

ai_suitability:
How useful AI assistance could be when performing the task.

Rules:
- Analyze every task.
- Preserve the task number.
- Do not invent additional tasks.
- Use only Low, Medium, or High for the three ratings.
- Return ONLY valid JSON.
- Do not use Markdown or code fences.
- Do not include explanations outside the JSON.

Return exactly this structure:

[
  {
    "task_number": 1,
    "automation_potential": "High",
    "human_dependency": "Medium",
    "ai_suitability": "High"
  }
]

Tasks to analyze:

""" + tasks_text

response = llm.invoke(impact_prompt)

content = response.content

if isinstance(content, list):
    content = "".join(
        item.get("text", "") if isinstance(item, dict) else str(item)
        for item in content
    )

content = content.strip()

# Remove Markdown code fences if present
if content.startswith("```"):
    content = content.replace("```json", "").replace("```", "").strip()

# Find the JSON array
start = content.find("[")
end = content.rfind("]")

if start == -1 or end == -1:
    raise ValueError(
        "The model did not return a valid JSON array.\n\n"
        + content
    )

content = content[start:end + 1]

impact_analysis = json.loads(content)

# Validate number of results
if len(impact_analysis) != len(df_tasks):
    raise ValueError(
        f"Expected {len(df_tasks)} analyses, "
        f"but received {len(impact_analysis)}."
    )

allowed_values = {"Low", "Medium", "High"}

# Validate every result
for item in impact_analysis:

    required_fields = {
        "task_number",
        "automation_potential",
        "human_dependency",
        "ai_suitability"
    }

    if set(item.keys()) != required_fields:
        raise ValueError(
            f"Unexpected fields returned: {item.keys()}"
        )

    for field in [
        "automation_potential",
        "human_dependency",
        "ai_suitability"
    ]:
        if item[field] not in allowed_values:
            raise ValueError(
                f"Invalid value '{item[field]}' for {field}"
            )

print("AI impact analysis completed successfully!")
print("Number of analyzed tasks:", len(impact_analysis))

for item in impact_analysis:
    print(
        f"Task {item['task_number']}: "
        f"Automation={item['automation_potential']}, "
        f"Human dependency={item['human_dependency']}, "
        f"AI suitability={item['ai_suitability']}"
    )

AI impact analysis completed successfully!
Number of analyzed tasks: 7
Task 1: Automation=High, Human dependency=Medium, AI suitability=High
Task 2: Automation=High, Human dependency=Medium, AI suitability=High
Task 3: Automation=Medium, Human dependency=Medium, AI suitability=High
Task 4: Automation=Medium, Human dependency=High, AI suitability=Medium
Task 5: Automation=Medium, Human dependency=High, AI suitability=Medium
Task 6: Automation=Low, Human dependency=High, AI suitability=Medium
Task 7: Automation=Low, Human dependency=High, AI suitability=Medium


In [ ]:
# Add AI impact analysis to the task DataFrame

df_tasks["automation_potential"] = [
    item["automation_potential"]
    for item in impact_analysis
]

df_tasks["human_dependency"] = [
    item["human_dependency"]
    for item in impact_analysis
]

df_tasks["ai_suitability"] = [
    item["ai_suitability"]
    for item in impact_analysis
]

print("AI impact analysis added to DataFrame!")
display(df_tasks)

AI impact analysis added to DataFrame!


,task,automation_potential,human_dependency,ai_suitability
0,Collect raw data from various internal and ext...,High,Medium,High
1,Clean and preprocess data to ensure accuracy a...,High,Medium,High
2,Perform statistical analysis to identify trend...,Medium,Medium,High
3,Develop and maintain interactive dashboards an...,Medium,High,Medium
4,Validate data quality and resolve inconsistencies,Medium,High,Medium
5,Collaborate with stakeholders to gather and cl...,Low,High,Medium
6,Present analytical findings and insights to su...,Low,High,Medium


In [ ]:
score_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

# Convert categories into numerical scores
df_tasks["automation_score"] = (
    df_tasks["automation_potential"].map(score_map)
)

df_tasks["human_dependency_score"] = (
    df_tasks["human_dependency"].map(score_map)
)

df_tasks["ai_suitability_score"] = (
    df_tasks["ai_suitability"].map(score_map)
)

# Calculate overall AI impact score
df_tasks["ai_impact_score"] = (
    df_tasks["automation_score"]
    + df_tasks["ai_suitability_score"]
    - df_tasks["human_dependency_score"]
)

display(df_tasks)

,task,automation_potential,human_dependency,ai_suitability,automation_score,human_dependency_score,ai_suitability_score,ai_impact_score
0,Collect raw data from various internal and ext...,High,Medium,High,3,2,3,4
1,Clean and preprocess data to ensure accuracy a...,High,Medium,High,3,2,3,4
2,Perform statistical analysis to identify trend...,Medium,Medium,High,2,2,3,3
3,Develop and maintain interactive dashboards an...,Medium,High,Medium,2,3,2,1
4,Validate data quality and resolve inconsistencies,Medium,High,Medium,2,3,2,1
5,Collaborate with stakeholders to gather and cl...,Low,High,Medium,1,3,2,0
6,Present analytical findings and insights to su...,Low,High,Medium,1,3,2,0


In [ ]:
df_ranked = (
    df_tasks
    .sort_values(
        by="ai_impact_score",
        ascending=False
    )
    .reset_index(drop=True)
)

df_ranked["ai_impact_rank"] = (
    df_ranked["ai_impact_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

print("Task ranking completed successfully!")
display(
    df_ranked[
        [
            "ai_impact_rank",
            "task",
            "ai_impact_score",
            "automation_potential",
            "human_dependency",
            "ai_suitability"
        ]
    ]
)

Task ranking completed successfully!


,ai_impact_rank,task,ai_impact_score,automation_potential,human_dependency,ai_suitability
0,1,Collect raw data from various internal and ext...,4,High,Medium,High
1,1,Clean and preprocess data to ensure accuracy a...,4,High,Medium,High
2,2,Perform statistical analysis to identify trend...,3,Medium,Medium,High
3,3,Develop and maintain interactive dashboards an...,1,Medium,High,Medium
4,3,Validate data quality and resolve inconsistencies,1,Medium,High,Medium
5,4,Collaborate with stakeholders to gather and cl...,0,Low,High,Medium
6,4,Present analytical findings and insights to su...,0,Low,High,Medium


In [ ]:
import json
import pandas as pd
from langchain_core.prompts import PromptTemplate

skill_prompt = PromptTemplate(
    input_variables=["task"],
    template="""
You are a job-skill analysis assistant.

Analyze the following job task and identify the skills required to perform it.

Rules:
- Identify 3 to 6 relevant skills.
- Include technical and/or professional skills when appropriate.
- Keep skills specific to the task.
- Identify the most important skills to focus on for future upskilling.
- Do not discuss AI impact.
- Do not assign scores.
- Return ONLY valid JSON.
- Do not use Markdown or code fences.

Return the JSON exactly in this format:

{{
    "skills": ["Skill 1", "Skill 2", "Skill 3"],
    "upskill_focus": ["Most important skill 1", "Most important skill 2"]
}}

Job task:
{task}
"""
)

skill_chain = skill_prompt | llm

skill_analysis = []

for task in df_tasks["task"]:
    response = skill_chain.invoke({"task": task})

    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in content
        )

    content = content.strip()

    if content.startswith("```"):
        content = content.replace("```json", "").replace("```", "").strip()

    analysis = json.loads(content)

    if "skills" not in analysis or "upskill_focus" not in analysis:
        raise ValueError(f"Invalid response for task: {task}")

    skill_analysis.append(analysis)

df_tasks["skills"] = [
    item["skills"] for item in skill_analysis
]

df_tasks["upskill_focus"] = [
    item["upskill_focus"] for item in skill_analysis
]

print("Skill analysis completed successfully!")
print("Number of analyzed tasks:", len(skill_analysis))

display(df_tasks[[
    "task",
    "skills",
    "upskill_focus"
]])

Skill analysis completed successfully!
Number of analyzed tasks: 7


,task,skills,upskill_focus
0,Collect raw data from various internal and ext...,"[Data acquisition, API integration, Web scrapi...","[Data acquisition, API integration]"
1,Clean and preprocess data to ensure accuracy a...,"[Data cleaning, Data preprocessing, Data valid...","[Data cleaning, Data validation and quality as..."
2,Perform statistical analysis to identify trend...,"[Statistical analysis, Data cleaning and prepr...","[Statistical analysis, Data visualization]"
3,Develop and maintain interactive dashboards an...,"[Data Visualization, Dashboard Development (e....","[Data Visualization, Dashboard Development (e...."
4,Validate data quality and resolve inconsistencies,"[Data Quality Assessment, Data Cleaning, SQL Q...","[Data Cleaning, SQL Querying]"
5,Collaborate with stakeholders to gather and cl...,"[Business Analysis, Requirements Elicitation, ...","[Requirements Elicitation, Stakeholder Communi..."
6,Present analytical findings and insights to su...,"[Data Analysis, Data Visualization, Presentati...","[Data Visualization, Presentation Skills]"


In [ ]:
from collections import Counter

# Combine all skills from all tasks
all_skills = []

for skills in df_tasks["skills"]:
    all_skills.extend(skills)

# Count how frequently each skill appears
skill_counts = Counter(all_skills)

# Create a DataFrame
skill_gap_df = pd.DataFrame(
    skill_counts.items(),
    columns=["skill", "task_frequency"]
)

# Sort by frequency
skill_gap_df = skill_gap_df.sort_values(
    by="task_frequency",
    ascending=False
).reset_index(drop=True)

print("Overall skill analysis completed successfully!")
print("Number of unique skills:", len(skill_gap_df))

display(skill_gap_df)

Overall skill analysis completed successfully!
Number of unique skills: 34


,skill,task_frequency
0,Data Analysis,2
1,Data Visualization,2
2,API integration,1
3,Data acquisition,1
4,Data management,1
5,Web scraping,1
6,Data preprocessing,1
7,Data validation and quality assurance,1
8,Python (pandas) for data manipulation,1
9,SQL querying,1


In [ ]:
# Normalize similar skill names into common categories

skill_normalization = {
    "data acquisition": "Data Acquisition",
    "api integration": "API Integration",
    "web scraping": "Web Scraping",
    "data management": "Data Management",
    "sql querying": "SQL",
    "sql querying": "SQL",
    "sql for data extraction": "SQL",
    "sql/data querying": "SQL",
    "data preprocessing": "Data Cleaning & Preprocessing",
    "data cleaning": "Data Cleaning & Preprocessing",
    "data cleansing": "Data Cleaning & Preprocessing",
    "data cleaning and preprocessing": "Data Cleaning & Preprocessing",
    "data validation": "Data Validation",
    "data quality assessment": "Data Quality",
    "python (pandas) for data manipulation": "Python (Pandas)",
    "programming (python or r)": "Programming (Python/R)",
    "statistical analysis": "Statistical Analysis",
    "knowledge of statistical modeling techniques": "Statistical Modeling",
    "data visualization": "Data Visualization",
    "dashboard development (e.g., power bi, tableau)": "Dashboard Development",
    "data modeling": "Data Modeling",
    "report automation (e.g., python, dax)": "Report Automation",
    "attention to detail": "Attention to Detail",
    "problem solving": "Problem Solving",
    "requirements elicitation": "Requirements Elicitation",
    "stakeholder management": "Stakeholder Management",
    "business analysis": "Business Analysis",
    "effective communication": "Communication",
    "active listening": "Active Listening",
    "data analysis": "Data Analysis",
    "presentation skills": "Presentation Skills"
}


def normalize_skill(skill):
    key = skill.strip().lower()
    return skill_normalization.get(key, skill.strip())


# Collect and normalize all skills
normalized_skills = []

for skills in df_tasks["skills"]:
    for skill in skills:
        normalized_skills.append(normalize_skill(skill))


# Count normalized skills
normalized_skill_counts = Counter(normalized_skills)


# Create final skill frequency DataFrame
skill_gap_df = pd.DataFrame(
    normalized_skill_counts.items(),
    columns=["skill", "task_frequency"]
)

skill_gap_df = skill_gap_df.sort_values(
    by="task_frequency",
    ascending=False
).reset_index(drop=True)


print("Skill normalization completed successfully!")
print("Number of unique normalized skills:", len(skill_gap_df))

display(skill_gap_df)

Skill normalization completed successfully!
Number of unique normalized skills: 27


,skill,task_frequency
0,SQL,4
1,Data Cleaning & Preprocessing,4
2,Data Visualization,3
3,Data Analysis,2
4,Web Scraping,1
5,Data Acquisition,1
6,API Integration,1
7,Data validation and quality assurance,1
8,Python (Pandas),1
9,Statistical Analysis,1


In [ ]:
# Select the most frequently required skills

top_skills = skill_gap_df.head(10).copy()

print("Top skills identified successfully!")
display(top_skills)

Top skills identified successfully!


,skill,task_frequency
0,SQL,4
1,Data Cleaning & Preprocessing,4
2,Data Visualization,3
3,Data Analysis,2
4,Web Scraping,1
5,Data Acquisition,1
6,API Integration,1
7,Data validation and quality assurance,1
8,Python (Pandas),1
9,Statistical Analysis,1


In [ ]:
# ============================================================
# STEP 13: VERIFY AND EXTRACT TOP 10 SKILLS
# ============================================================

print("=" * 60)
print("TOP 10 SKILLS FOR FURTHER ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Validate top_skills
# ------------------------------------------------------------

if not isinstance(top_skills, pd.DataFrame):
    raise TypeError(
        f"top_skills should be a pandas DataFrame, "
        f"but found: {type(top_skills)}"
    )

if "skill" not in top_skills.columns:
    raise KeyError(
        f"'skill' column not found in top_skills.\n"
        f"Available columns: {list(top_skills.columns)}"
    )

# ------------------------------------------------------------
# Extract the ACTUAL skill names
# ------------------------------------------------------------

final_top_skills = (
    top_skills["skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .head(10)
    .tolist()
)

# ------------------------------------------------------------
# Validate exactly 10 skills
# ------------------------------------------------------------

print(f"\nTotal skills selected: {len(final_top_skills)}\n")

for i, skill in enumerate(final_top_skills, start=1):
    print(f"{i}. {skill}")

if len(final_top_skills) != 10:
    raise ValueError(
        f"Expected exactly 10 skills, "
        f"but found {len(final_top_skills)}.\n"
        f"Check the top_skills DataFrame above."
    )

print("\n✓ Exactly 10 skills confirmed.")

TOP 10 SKILLS FOR FURTHER ANALYSIS

Total skills selected: 10

1. SQL
2. Data Cleaning & Preprocessing
3. Data Visualization
4. Data Analysis
5. Web Scraping
6. Data Acquisition
7. API Integration
8. Data validation and quality assurance
9. Python (Pandas)
10. Statistical Analysis

✓ Exactly 10 skills confirmed.


In [ ]:
# ============================================================
# STEP 14: VERIFIED LEARNING RESOURCES
# ============================================================

print("=" * 60)
print("STEP 14: VERIFIED LEARNING RESOURCES")
print("=" * 60)

# ------------------------------------------------------------
# Official / verified learning resources
# ------------------------------------------------------------
#
# These URLs were checked against the official websites:
#
# Microsoft Learn
# PostgreSQL official documentation
# Great Expectations official documentation
#
# ------------------------------------------------------------

verified_resources = {

    "SQL": {
        "source": "PostgreSQL Official Documentation",
        "learning_websites": "PostgreSQL",
        "learning_links": (
            "https://www.postgresql.org/docs/current/tutorial.html"
        ),
        "reason": (
            "Official PostgreSQL tutorial covering SQL fundamentals, "
            "querying, joins, aggregates, updates and related database concepts."
        )
    },

    "Data Analysis": {
        "source": "Microsoft Learn",
        "learning_websites": "Microsoft Learn",
        "learning_links": (
            "https://learn.microsoft.com/en-us/training/career-paths/data-analyst"
        ),
        "reason": (
            "Official Microsoft learning path for data analysts covering "
            "data analysis, preparation, transformation, visualization and reporting."
        )
    },

    "Data Cleaning & Preprocessing": {
        "source": "Microsoft Learn",
        "learning_websites": "Microsoft Learn",
        "learning_links": (
            "https://learn.microsoft.com/en-us/training/career-paths/data-analyst"
        ),
        "reason": (
            "Microsoft's official Data Analyst learning path includes "
            "profiling, cleaning and transforming data."
        )
    },

    "Data Visualization": {
        "source": "Microsoft Learn",
        "learning_websites": "Microsoft Learn",
        "learning_links": (
            "https://learn.microsoft.com/en-us/training/paths/data-analytics-microsoft/"
        ),
        "reason": (
            "Official Microsoft data analytics training covering "
            "reports, dashboards and visualization with Power BI."
        )
    },

    "Database querying": {
        "source": "PostgreSQL Official Documentation",
        "learning_websites": "PostgreSQL",
        "learning_links": (
            "https://www.postgresql.org/docs/current/tutorial-sql.html"
        ),
        "reason": (
            "Official PostgreSQL SQL documentation covering querying tables, "
            "joins, aggregates and other SQL operations."
        )
    },

    "Data governance": {
        "source": "Microsoft Learn",
        "learning_websites": "Microsoft Learn",
        "learning_links": (
            "https://learn.microsoft.com/en-us/training/paths/explore-data-governance-microsoft-365/"
        ),
        "reason": (
            "Official Microsoft learning path introducing data governance, "
            "data integrity and governance-related practices."
        )
    },

    "Data validation and quality assurance": {
        "source": "Great Expectations Official Documentation",
        "learning_websites": "Great Expectations",
        "learning_links": (
            "https://docs.greatexpectations.io/docs/reference/learn/data_quality_use_cases/integrity/"
        ),
        "reason": (
            "Official Great Expectations documentation covering data integrity, "
            "quality checks and validation practices."
        )
    },

    "Data Acquisition": {
        "source": "Microsoft Learn",
        "learning_websites": "Microsoft Learn",
        "learning_links": (
            "https://learn.microsoft.com/en-us/training/career-paths/data-analyst"
        ),
        "reason": (
            "Official Microsoft Data Analyst learning path covering "
            "identifying appropriate data and preparing data for analysis."
        )
    },

    "API Integration": {
        "source": "Python Requests Official Documentation",
        "learning_websites": "Requests",
        "learning_links": (
            "https://requests.readthedocs.io/en/latest/"
        ),
        "reason": (
            "Official Requests documentation for making HTTP requests "
            "and interacting with web APIs from Python."
        )
    },

    "Data Management": {
        "source": "PostgreSQL Official Documentation",
        "learning_websites": "PostgreSQL",
        "learning_links": (
            "https://www.postgresql.org/docs/current/tutorial.html"
        ),
        "reason": (
            "Official PostgreSQL tutorial covering relational databases, "
            "tables, queries and database operations."
        )
    },

    "Web Scraping": {
        "source": "Beautiful Soup Official Documentation",
        "learning_websites": "Beautiful Soup",
        "learning_links": (
            "https://beautiful-soup-4.readthedocs.io/en/latest/"
        ),
        "reason": (
            "Official Beautiful Soup documentation for extracting data "
            "from HTML and XML documents."
        )
    },

    "Python (Pandas)": {
        "source": "pandas Official Documentation",
        "learning_websites": "pandas",
        "learning_links": (
            "https://pandas.pydata.org/docs/getting_started/"
        ),
        "reason": (
            "Official pandas documentation for DataFrames, data loading, "
            "cleaning, manipulation and analysis."
        )
    },

    "Statistical Analysis": {
        "source": "SciPy Official Documentation",
        "learning_websites": "SciPy",
        "learning_links": (
            "https://docs.scipy.org/doc/scipy/tutorial/stats.html"
        ),
        "reason": (
            "Official SciPy statistics documentation covering statistical "
            "methods and probability distributions."
        )
    }
}


# ------------------------------------------------------------
# Required skills
# ------------------------------------------------------------

print("\nChecking required skills...")

# Use the skills that actually exist in the current notebook.
# If one of these variables exists, use it.
if "research_skills" in globals():

    current_skills = list(research_skills)

elif "final_top_skills" in globals():

    current_skills = list(final_top_skills)

elif "top_skills" in globals():

    if isinstance(top_skills, pd.DataFrame):
        if "skill" not in top_skills.columns:
            raise KeyError(
                "top_skills DataFrame does not contain a 'skill' column."
            )

        current_skills = (
            top_skills["skill"]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )

    else:
        current_skills = list(top_skills)

else:
    raise NameError(
        "Could not find research_skills, final_top_skills, "
        "or top_skills."
    )


# ------------------------------------------------------------
# Normalize skill names
# ------------------------------------------------------------

current_skills = [
    str(skill).strip()
    for skill in current_skills
    if str(skill).strip()
]


# Remove duplicates while preserving order
current_skills = list(dict.fromkeys(current_skills))


print(f"Skills requiring verified resources: {len(current_skills)}")

for i, skill in enumerate(current_skills, 1):
    print(f"{i}. {skill}")


# ------------------------------------------------------------
# Find missing resources
# ------------------------------------------------------------

missing_skills = [
    skill
    for skill in current_skills
    if skill not in verified_resources
]


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if missing_skills:

    print("\n❌ Missing verified resources:")
    for skill in missing_skills:
        print(f"- {skill}")

    raise ValueError(
        f"Missing verified resources for: {missing_skills}"
    )


# ------------------------------------------------------------
# Build final resource dataframe
# ------------------------------------------------------------

resource_rows = []

for skill in current_skills:

    resource = verified_resources[skill]

    resource_rows.append({
        "skill": skill,
        "source": resource["source"],
        "learning_websites": resource["learning_websites"],
        "learning_links": resource["learning_links"],
        "reason": resource["reason"]
    })


web_skills_df = pd.DataFrame(resource_rows)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if len(web_skills_df) != len(current_skills):

    raise ValueError(
        "Resource dataframe size does not match required skills."
    )


if not web_skills_df["learning_links"].astype(str).str.startswith(
    "http"
).all():

    raise ValueError(
        "One or more learning links are invalid."
    )


if web_skills_df["learning_links"].astype(str).str.len().eq(0).any():

    raise ValueError(
        "One or more learning links are empty."
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VERIFIED LEARNING RESOURCES")
print("=" * 60)

print(
    f"\nVerified resources created successfully!"
)

print(
    f"Number of skills: {len(web_skills_df)}"
)

display(web_skills_df)


print("\n" + "=" * 60)
print("✓ STEP 14 COMPLETED SUCCESSFULLY.")
print("=" * 60)

STEP 14: VERIFIED LEARNING RESOURCES

Checking required skills...
Skills requiring verified resources: 10
1. SQL
2. Data Cleaning & Preprocessing
3. Data Visualization
4. Data Analysis
5. Web Scraping
6. Data Acquisition
7. API Integration
8. Data validation and quality assurance
9. Python (Pandas)
10. Statistical Analysis

VERIFIED LEARNING RESOURCES

Verified resources created successfully!
Number of skills: 10


,skill,source,learning_websites,learning_links,reason
0,SQL,PostgreSQL Official Documentation,PostgreSQL,https://www.postgresql.org/docs/current/tutori...,Official PostgreSQL tutorial covering SQL fund...
1,Data Cleaning & Preprocessing,Microsoft Learn,Microsoft Learn,https://learn.microsoft.com/en-us/training/car...,Microsoft's official Data Analyst learning pat...
2,Data Visualization,Microsoft Learn,Microsoft Learn,https://learn.microsoft.com/en-us/training/pat...,Official Microsoft data analytics training cov...
3,Data Analysis,Microsoft Learn,Microsoft Learn,https://learn.microsoft.com/en-us/training/car...,Official Microsoft learning path for data anal...
4,Web Scraping,Beautiful Soup Official Documentation,Beautiful Soup,https://beautiful-soup-4.readthedocs.io/en/lat...,Official Beautiful Soup documentation for extr...
5,Data Acquisition,Microsoft Learn,Microsoft Learn,https://learn.microsoft.com/en-us/training/car...,Official Microsoft Data Analyst learning path ...
6,API Integration,Python Requests Official Documentation,Requests,https://requests.readthedocs.io/en/latest/,Official Requests documentation for making HTT...
7,Data validation and quality assurance,Great Expectations Official Documentation,Great Expectations,https://docs.greatexpectations.io/docs/referen...,Official Great Expectations documentation cove...
8,Python (Pandas),pandas Official Documentation,pandas,https://pandas.pydata.org/docs/getting_started/,"Official pandas documentation for DataFrames, ..."
9,Statistical Analysis,SciPy Official Documentation,SciPy,https://docs.scipy.org/doc/scipy/tutorial/stat...,Official SciPy statistics documentation coveri...



✓ STEP 14 COMPLETED SUCCESSFULLY.


In [ ]:
import os

print("Checking O*NET files...\n")

files_to_check = [
    "occupation_data.csv",
    "software_skills.csv",
    "task_ratings.csv",
    "task_statements.csv"
]

for filename in files_to_check:
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"✅ {filename:<25} {size_mb:.2f} MB")
    else:
        print(f"❌ {filename:<25} NOT FOUND")

Checking O*NET files...

✅ occupation_data.csv       0.26 MB
✅ software_skills.csv       3.39 MB
✅ task_ratings.csv          37.20 MB
✅ task_statements.csv       3.30 MB


In [ ]:
# ============================================================
# STEP 15: LOAD AND VERIFY O*NET 31.0 DATA
# ============================================================

import os
import pandas as pd

print("=" * 60)
print("STEP 15: LOADING O*NET 31.0 DATA")
print("=" * 60)

# Required O*NET files
required_files = {
    "occupation_data": "occupation_data.csv",
    "task_statements": "task_statements.csv",
    "task_ratings": "task_ratings.csv",
    "software_skills": "software_skills.csv"
}

# ------------------------------------------------------------
# CHECK FILES
# ------------------------------------------------------------

missing_files = []

for name, filename in required_files.items():
    if not os.path.exists(filename):
        missing_files.append(filename)

if missing_files:
    print("\n❌ MISSING FILES:")
    for filename in missing_files:
        print(f"   - {filename}")

    raise FileNotFoundError(
        "\nUpload all 4 required O*NET CSV files to Colab "
        "and run this cell again."
    )

print("\n✅ All required O*NET files found.")

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

occupation_data = pd.read_csv(
    "occupation_data.csv",
    low_memory=False
)

task_statements = pd.read_csv(
    "task_statements.csv",
    low_memory=False
)

task_ratings = pd.read_csv(
    "task_ratings.csv",
    low_memory=False
)

software_skills = pd.read_csv(
    "software_skills.csv",
    low_memory=False
)

# ------------------------------------------------------------
# DISPLAY DATASET SIZES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASETS LOADED")
print("=" * 60)

print(f"\nOccupation Data : {occupation_data.shape}")
print(f"Task Statements : {task_statements.shape}")
print(f"Task Ratings    : {task_ratings.shape}")
print(f"Software Skills : {software_skills.shape}")

# ------------------------------------------------------------
# DISPLAY COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("COLUMN VERIFICATION")
print("=" * 60)

for name, df in {
    "Occupation Data": occupation_data,
    "Task Statements": task_statements,
    "Task Ratings": task_ratings,
    "Software Skills": software_skills
}.items():

    print(f"\n{name}:")
    print(df.columns.tolist())

# ------------------------------------------------------------
# SUCCESS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✅ O*NET DATA LOADED SUCCESSFULLY")
print("=" * 60)

STEP 15: LOADING O*NET 31.0 DATA

✅ All required O*NET files found.

DATASETS LOADED

Occupation Data : (1016, 3)
Task Statements : (18838, 8)
Task Ratings    : (165780, 15)
Software Skills : (31821, 7)

COLUMN VERIFICATION

Occupation Data:
['O*NET-SOC Code', 'Title', 'Description']

Task Statements:
['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']

Task Ratings:
['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Scale ID', 'Scale Name', 'Category', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Date', 'Domain Source']

Software Skills:
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']

✅ O*NET DATA LOADED SUCCESSFULLY


In [ ]:
# ============================================================
# STEP 16: IDENTIFY THE O*NET OCCUPATION FOR DATA ANALYST WORK
# ============================================================

print("=" * 60)
print("STEP 16: IDENTIFYING O*NET ANALYST OCCUPATION")
print("=" * 60)

# O*NET does not contain an occupation literally named
# "Data Analyst" in this dataset.
#
# For this project, we use the O*NET occupation:
# Business Intelligence Analysts
# O*NET-SOC Code: 15-2051.01

TARGET_OCCUPATION_CODE = "15-2051.01"
TARGET_OCCUPATION_TITLE = "Business Intelligence Analysts"

# Find the occupation using the official O*NET-SOC code
analyst_occupation = occupation_data[
    occupation_data["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
    .eq(TARGET_OCCUPATION_CODE)
].copy()

# Safety check
if analyst_occupation.empty:
    raise ValueError(
        f"O*NET occupation code {TARGET_OCCUPATION_CODE} "
        "was not found in occupation_data."
    )

# Display the matched occupation
print("\nMatched O*NET occupation:")
display(analyst_occupation)

print("\n" + "=" * 60)
print("O*NET OCCUPATION CONFIRMED")
print("=" * 60)

print(f"Occupation Code : {TARGET_OCCUPATION_CODE}")
print(f"Occupation Title: {TARGET_OCCUPATION_TITLE}")

print("\n✓ O*NET occupation successfully identified.")

STEP 16: IDENTIFYING O*NET ANALYST OCCUPATION

Matched O*NET occupation:


,O*NET-SOC Code,Title,Description
143,15-2051.01,Business Intelligence Analysts,Produce financial and market intelligence by q...



O*NET OCCUPATION CONFIRMED
Occupation Code : 15-2051.01
Occupation Title: Business Intelligence Analysts

✓ O*NET occupation successfully identified.


In [ ]:
# ============================================================
# STEP 17: EXTRACT TASKS FOR BUSINESS INTELLIGENCE ANALYSTS
# ============================================================

print("=" * 60)
print("STEP 17: EXTRACTING OCCUPATION-SPECIFIC TASKS")
print("=" * 60)

# Use the confirmed O*NET-SOC code from Step 16
TARGET_CODE = TARGET_OCCUPATION_CODE

# Filter task statements for the target occupation
analyst_tasks = task_statements[
    task_statements["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
    .eq(TARGET_CODE)
].copy()

# Safety check
if analyst_tasks.empty:
    raise ValueError(
        f"No task statements found for O*NET-SOC code {TARGET_CODE}."
    )

# Reset index for clean analysis
analyst_tasks.reset_index(drop=True, inplace=True)

print(f"\nOccupation: {TARGET_OCCUPATION_TITLE}")
print(f"O*NET-SOC Code: {TARGET_CODE}")
print(f"Number of task statements found: {len(analyst_tasks)}")

print("\n" + "=" * 60)
print("TASK DATA PREVIEW")
print("=" * 60)

# Display only the most useful columns
task_preview_columns = [
    "O*NET-SOC Code",
    "Title",
    "Task ID",
    "Task",
    "Task Type"
]

display(analyst_tasks[task_preview_columns].head(15))

print("\n✓ Occupation-specific task data extracted successfully.")

STEP 17: EXTRACTING OCCUPATION-SPECIFIC TASKS

Occupation: Business Intelligence Analysts
O*NET-SOC Code: 15-2051.01
Number of task statements found: 17

TASK DATA PREVIEW


,O*NET-SOC Code,Title,Task ID,Task,Task Type
0,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Core
1,15-2051.01,Business Intelligence Analysts,16139,Maintain or update business intelligence tools...,Core
2,15-2051.01,Business Intelligence Analysts,16140,Manage timely flow of business intelligence in...,Core
3,15-2051.01,Business Intelligence Analysts,16134,Provide technical support for existing reports...,Core
4,15-2051.01,Business Intelligence Analysts,16141,Identify and analyze industry or geographic tr...,Core
5,15-2051.01,Business Intelligence Analysts,16142,Document specifications for business intellige...,Core
6,15-2051.01,Business Intelligence Analysts,16144,"Create business intelligence tools or systems,...",Core
7,15-2051.01,Business Intelligence Analysts,16146,Collect business intelligence data from availa...,Core
8,15-2051.01,Business Intelligence Analysts,16143,"Disseminate information regarding tools, repor...",Core
9,15-2051.01,Business Intelligence Analysts,16145,Conduct or coordinate tests to ensure that int...,Core



✓ Occupation-specific task data extracted successfully.


In [ ]:
# ============================================================
# STEP 18: MATCH TASK RATINGS TO ANALYST TASKS
# ============================================================

print("=" * 60)
print("STEP 18: MATCHING TASK RATINGS TO ANALYST TASKS")
print("=" * 60)

# Get the Task IDs belonging to our confirmed occupation
analyst_task_ids = (
    analyst_tasks["Task ID"]
    .astype(str)
    .str.strip()
    .unique()
)

print(f"\nOccupation: {TARGET_OCCUPATION_TITLE}")
print(f"O*NET-SOC Code: {TARGET_CODE}")
print(f"Unique task IDs: {len(analyst_task_ids)}")

# Filter task ratings using both:
# 1. Occupation code
# 2. Task ID
analyst_task_ratings = task_ratings[
    task_ratings["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
    .eq(TARGET_CODE)
    &
    task_ratings["Task ID"]
    .astype(str)
    .str.strip()
    .isin(analyst_task_ids)
].copy()

# Reset index
analyst_task_ratings.reset_index(drop=True, inplace=True)

print(f"Task-rating records matched: {len(analyst_task_ratings)}")

# Safety check
if analyst_task_ratings.empty:
    raise ValueError(
        "No task-rating records were matched to the analyst tasks."
    )

print("\n" + "=" * 60)
print("TASK RATING COLUMNS")
print("=" * 60)

print(list(analyst_task_ratings.columns))

print("\n" + "=" * 60)
print("TASK RATINGS PREVIEW")
print("=" * 60)

display(
    analyst_task_ratings[
        [
            "O*NET-SOC Code",
            "Title",
            "Task ID",
            "Task",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ].head(20)
)

print("\n✓ Task ratings matched successfully.")

STEP 18: MATCHING TASK RATINGS TO ANALYST TASKS

Occupation: Business Intelligence Analysts
O*NET-SOC Code: 15-2051.01
Unique task IDs: 17
Task-rating records matched: 153

TASK RATING COLUMNS
['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Scale ID', 'Scale Name', 'Category', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Date', 'Domain Source']

TASK RATINGS PREVIEW


,O*NET-SOC Code,Title,Task ID,Task,Scale Name,Category,Data Value
0,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),1.0,0.00
1,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),2.0,4.55
2,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),3.0,13.64
3,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),4.0,31.82
4,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),5.0,18.18
5,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),6.0,13.64
6,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Frequency of Task (Categories 1-7),7.0,18.18
7,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Importance,NaN,4.64
8,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,Relevance of Task,NaN,100.00
9,15-2051.01,Business Intelligence Analysts,16139,Maintain or update business intelligence tools...,Frequency of Task (Categories 1-7),1.0,0.00



✓ Task ratings matched successfully.


In [ ]:
# ============================================================
# STEP 19: CALCULATING TASK-LEVEL METRICS
# ============================================================

print("\n" + "=" * 60)
print("STEP 19: CALCULATING TASK-LEVEL METRICS")
print("=" * 60)

# ------------------------------------------------------------
# IMPORTANT:
# Restrict task ratings to the TARGET occupation first.
# Otherwise ratings from every O*NET occupation get included.
# ------------------------------------------------------------

target_ratings = task_ratings[
    task_ratings["O*NET-SOC Code"] == TARGET_OCCUPATION_CODE
].copy()

print(f"\nOccupation: {TARGET_OCCUPATION_TITLE}")
print(f"O*NET-SOC Code: {TARGET_OCCUPATION_CODE}")
print(f"Task-rating records for target occupation: {len(target_ratings)}")

# ------------------------------------------------------------
# Convert Data Value to numeric
# ------------------------------------------------------------

target_ratings["Data Value"] = pd.to_numeric(
    target_ratings["Data Value"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. IMPORTANCE
# ------------------------------------------------------------

importance_data = target_ratings[
    target_ratings["Scale Name"].str.contains(
        "Importance",
        case=False,
        na=False
    )
].copy()

importance_data = (
    importance_data[
        [
            "O*NET-SOC Code",
            "Title",
            "Task ID",
            "Task",
            "Data Value"
        ]
    ]
    .rename(columns={"Data Value": "Importance"})
)

# ------------------------------------------------------------
# 2. RELEVANCE
# ------------------------------------------------------------

relevance_data = target_ratings[
    target_ratings["Scale Name"].str.contains(
        "Relevance of Task",
        case=False,
        na=False
    )
].copy()

relevance_data = (
    relevance_data[
        [
            "Task ID",
            "Data Value"
        ]
    ]
    .rename(columns={"Data Value": "Relevance"})
)

# ------------------------------------------------------------
# 3. FREQUENCY
#
# O*NET Frequency of Task contains categories 1–7.
# Data Value represents the percentage of responses in each
# category.
#
# We calculate a weighted frequency score:
#
#   sum(category × percentage) / 100
#
# This gives one frequency score for each task.
# ------------------------------------------------------------

frequency_data = target_ratings[
    target_ratings["Scale Name"].str.contains(
        "Frequency of Task",
        case=False,
        na=False
    )
].copy()

frequency_data["Category"] = pd.to_numeric(
    frequency_data["Category"],
    errors="coerce"
)

frequency_data["Data Value"] = pd.to_numeric(
    frequency_data["Data Value"],
    errors="coerce"
)

frequency_data = frequency_data.dropna(
    subset=["Task ID", "Category", "Data Value"]
)

frequency_scores = (
    frequency_data
    .assign(
        Weighted_Value=
        frequency_data["Category"] *
        frequency_data["Data Value"]
    )
    .groupby("Task ID", as_index=False)
    .agg(
        Weighted_Sum=("Weighted_Value", "sum"),
        Percentage_Sum=("Data Value", "sum")
    )
)

frequency_scores["Frequency"] = (
    frequency_scores["Weighted_Sum"] /
    frequency_scores["Percentage_Sum"]
)

frequency_scores = frequency_scores[
    ["Task ID", "Frequency"]
]

# ------------------------------------------------------------
# MERGE ALL THREE METRICS
# ------------------------------------------------------------

task_level_metrics = (
    importance_data
    .merge(
        relevance_data,
        on="Task ID",
        how="left"
    )
    .merge(
        frequency_scores,
        on="Task ID",
        how="left"
    )
)

# ------------------------------------------------------------
# Clean and round
# ------------------------------------------------------------

task_level_metrics["Frequency"] = (
    pd.to_numeric(
        task_level_metrics["Frequency"],
        errors="coerce"
    ).round(2)
)

task_level_metrics["Importance"] = (
    pd.to_numeric(
        task_level_metrics["Importance"],
        errors="coerce"
    ).round(2)
)

task_level_metrics["Relevance"] = (
    pd.to_numeric(
        task_level_metrics["Relevance"],
        errors="coerce"
    ).round(2)
)

# ------------------------------------------------------------
# Sort by Importance
# ------------------------------------------------------------

task_level_metrics = (
    task_level_metrics
    .sort_values(
        by=["Importance", "Frequency"],
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TASK-LEVEL METRICS")
print("=" * 60)

print(f"\nNumber of unique tasks: {len(task_level_metrics)}")

display(task_level_metrics)

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

if len(task_level_metrics) != 17:
    raise ValueError(
        f"Expected 17 unique tasks, "
        f"but found {len(task_level_metrics)}."
    )

required_columns = [
    "O*NET-SOC Code",
    "Title",
    "Task ID",
    "Task",
    "Frequency",
    "Importance",
    "Relevance"
]

missing_columns = [
    col for col in required_columns
    if col not in task_level_metrics.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if task_level_metrics["Task ID"].nunique() != 17:
    raise ValueError(
        "Task ID uniqueness check failed."
    )

print("\n✓ Target occupation successfully isolated.")
print("✓ Frequency calculated from O*NET frequency categories.")
print("✓ Importance extracted.")
print("✓ Relevance extracted.")
print("✓ Exactly 17 unique analyst tasks confirmed.")
print("\n✓ STEP 19 COMPLETED SUCCESSFULLY.")


STEP 19: CALCULATING TASK-LEVEL METRICS

Occupation: Business Intelligence Analysts
O*NET-SOC Code: 15-2051.01
Task-rating records for target occupation: 153

TASK-LEVEL METRICS

Number of unique tasks: 17


,O*NET-SOC Code,Title,Task ID,Task,Importance,Relevance,Frequency
0,15-2051.01,Business Intelligence Analysts,16150,Generate standard or custom reports summarizin...,4.64,100.00,4.77
1,15-2051.01,Business Intelligence Analysts,16139,Maintain or update business intelligence tools...,4.36,100.00,4.68
2,15-2051.01,Business Intelligence Analysts,16140,Manage timely flow of business intelligence in...,4.19,100.00,4.81
3,15-2051.01,Business Intelligence Analysts,16134,Provide technical support for existing reports...,3.95,95.24,4.25
4,15-2051.01,Business Intelligence Analysts,16141,Identify and analyze industry or geographic tr...,3.86,100.00,3.50
5,15-2051.01,Business Intelligence Analysts,16142,Document specifications for business intellige...,3.73,100.00,3.41
6,15-2051.01,Business Intelligence Analysts,16144,"Create business intelligence tools or systems,...",3.70,95.24,3.70
7,15-2051.01,Business Intelligence Analysts,16146,Collect business intelligence data from availa...,3.68,100.00,3.59
8,15-2051.01,Business Intelligence Analysts,16143,"Disseminate information regarding tools, repor...",3.65,95.24,3.55
9,15-2051.01,Business Intelligence Analysts,16145,Conduct or coordinate tests to ensure that int...,3.64,100.00,3.32



✓ Target occupation successfully isolated.
✓ Frequency calculated from O*NET frequency categories.
✓ Importance extracted.
✓ Relevance extracted.
✓ Exactly 17 unique analyst tasks confirmed.

✓ STEP 19 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 20: RANKING ANALYST TASKS
# ============================================================

print("=" * 60)
print("STEP 20: RANKING ANALYST TASKS")
print("=" * 60)

# Work on a copy
ranked_tasks = task_level_metrics.copy()

# ------------------------------------------------------------
# 1. Normalize the three task metrics to 0-1
# ------------------------------------------------------------

for col in ["Importance", "Relevance", "Frequency"]:
    min_val = ranked_tasks[col].min()
    max_val = ranked_tasks[col].max()

    if max_val > min_val:
        ranked_tasks[f"{col}_Normalized"] = (
            (ranked_tasks[col] - min_val) /
            (max_val - min_val)
        )
    else:
        ranked_tasks[f"{col}_Normalized"] = 0.0

# ------------------------------------------------------------
# 2. Calculate overall task priority
#
# Importance gets the highest weight because it represents
# how important the task is to the occupation.
# ------------------------------------------------------------

ranked_tasks["Task_Priority"] = (
    ranked_tasks["Importance_Normalized"] * 0.50 +
    ranked_tasks["Relevance_Normalized"] * 0.30 +
    ranked_tasks["Frequency_Normalized"] * 0.20
)

# ------------------------------------------------------------
# 3. Rank tasks
# ------------------------------------------------------------

ranked_tasks = ranked_tasks.sort_values(
    by="Task_Priority",
    ascending=False
).reset_index(drop=True)

ranked_tasks["Task_Rank"] = ranked_tasks.index + 1

# ------------------------------------------------------------
# 4. Display important columns
# ------------------------------------------------------------

task_ranking_view = ranked_tasks[
    [
        "Task_Rank",
        "Task ID",
        "Task",
        "Importance",
        "Relevance",
        "Frequency",
        "Task_Priority"
    ]
]

display(task_ranking_view)

# ------------------------------------------------------------
# 5. Safety checks
# ------------------------------------------------------------

if len(ranked_tasks) != 17:
    raise ValueError(
        f"Expected 17 ranked tasks, but found {len(ranked_tasks)}."
    )

if ranked_tasks["Task_Priority"].isna().any():
    raise ValueError("Task priority contains missing values.")

print("\n✓ All 17 analyst tasks ranked successfully.")
print("✓ Task priority calculated from Importance, Relevance and Frequency.")
print("✓ STEP 20 COMPLETED SUCCESSFULLY.")

STEP 20: RANKING ANALYST TASKS


,Task_Rank,Task ID,Task,Importance,Relevance,Frequency,Task_Priority
0,1,16150,Generate standard or custom reports summarizin...,4.64,100.00,4.77,0.996172
1,2,16139,Maintain or update business intelligence tools...,4.36,100.00,4.68,0.888968
2,3,16140,Manage timely flow of business intelligence in...,4.19,100.00,4.81,0.841549
3,4,16134,Provide technical support for existing reports...,3.95,95.24,4.25,0.624906
4,5,16141,Identify and analyze industry or geographic tr...,3.86,100.00,3.50,0.599993
5,6,16142,Document specifications for business intellige...,3.73,100.00,3.41,0.545606
6,7,16146,Collect business intelligence data from availa...,3.68,100.00,3.59,0.545225
7,8,16145,Conduct or coordinate tests to ensure that int...,3.64,100.00,3.32,0.505304
8,9,16149,Synthesize current business intelligence or tr...,3.59,100.00,3.45,0.500138
9,10,16144,"Create business intelligence tools or systems,...",3.70,95.24,3.70,0.484246



✓ All 17 analyst tasks ranked successfully.
✓ Task priority calculated from Importance, Relevance and Frequency.
✓ STEP 20 COMPLETED SUCCESSFULLY.


In [ ]:
print("\n" + "=" * 70)
print("STEP 21: EXTRACTING OCCUPATION-SPECIFIC SOFTWARE SKILLS")
print("=" * 70)

# Filter software skills for target occupation
analyst_software_skills = software_skills[
    software_skills["O*NET-SOC Code"] == TARGET_OCCUPATION_CODE
].copy()

print(f"\nOccupation: {TARGET_OCCUPATION_TITLE}")
print(f"O*NET-SOC Code: {TARGET_OCCUPATION_CODE}")
print(f"Software-skill records found: {len(analyst_software_skills)}")

# Safety check
if analyst_software_skills.empty:
    raise ValueError(
        f"No software skills found for {TARGET_OCCUPATION_CODE}."
    )

print("\nSOFTWARE SKILL COLUMNS")
print("=" * 70)

print(analyst_software_skills.columns.tolist())

print("\nSOFTWARE SKILLS PREVIEW")
print("=" * 70)

display(
    analyst_software_skills[
        [
            "O*NET-SOC Code",
            "Title",
            "Workplace Example",
            "Element Name",
            "Hot Technology",
            "In Demand"
        ]
    ].head(20)
)

print("\n✓ Occupation-specific software skills extracted successfully.")


STEP 21: EXTRACTING OCCUPATION-SPECIFIC SOFTWARE SKILLS

Occupation: Business Intelligence Analysts
O*NET-SOC Code: 15-2051.01
Software-skill records found: 203

SOFTWARE SKILL COLUMNS
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']

SOFTWARE SKILLS PREVIEW


,O*NET-SOC Code,Title,Workplace Example,Element Name,Hot Technology,In Demand
11952,15-2051.01,Business Intelligence Analysts,Actuate Eclipse BIRT,Data base reporting software,N,N
11953,15-2051.01,Business Intelligence Analysts,Adobe Acrobat,Document management software,Y,N
11954,15-2051.01,Business Intelligence Analysts,Adobe ActionScript,Development environment software,N,N
11955,15-2051.01,Business Intelligence Analysts,Adobe Dreamweaver,Web page creation and editing software,N,N
11956,15-2051.01,Business Intelligence Analysts,Advanced business application programming ABAP,Object or component oriented development software,N,N
11957,15-2051.01,Business Intelligence Analysts,Airtable,Data base user interface and query software,N,N
11958,15-2051.01,Business Intelligence Analysts,AJAX,Web platform development software,Y,N
11959,15-2051.01,Business Intelligence Analysts,Alteryx software,Business intelligence and data analysis software,Y,N
11960,15-2051.01,Business Intelligence Analysts,Amazon DynamoDB,Data base management system software,Y,N
11961,15-2051.01,Business Intelligence Analysts,Amazon Elastic Compute Cloud EC2,Data base user interface and query software,Y,N



✓ Occupation-specific software skills extracted successfully.


In [ ]:
print("\n" + "=" * 70)
print("STEP 22: ANALYZING SOFTWARE TECHNOLOGY DEMAND")
print("=" * 70)

# Work on a copy
software_analysis = analyst_software_skills.copy()

# Clean key fields
software_analysis["Workplace Example"] = (
    software_analysis["Workplace Example"]
    .astype(str)
    .str.strip()
)

software_analysis["Element Name"] = (
    software_analysis["Element Name"]
    .astype(str)
    .str.strip()
)

# Remove empty / invalid technology names
software_analysis = software_analysis[
    software_analysis["Workplace Example"].notna()
    & (software_analysis["Workplace Example"] != "")
    & (software_analysis["Workplace Example"].str.lower() != "nan")
].copy()

# Normalize Y/N fields
software_analysis["Hot Technology"] = (
    software_analysis["Hot Technology"]
    .astype(str)
    .str.strip()
    .str.upper()
)

software_analysis["In Demand"] = (
    software_analysis["In Demand"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Create technology-level summary
technology_summary = (
    software_analysis
    .groupby("Workplace Example", as_index=False)
    .agg(
        software_category=("Element Name", "first"),
        hot_technology=("Hot Technology", lambda x: (x == "Y").any()),
        in_demand=("In Demand", lambda x: (x == "Y").any())
    )
)

# Convert Boolean values to integers
technology_summary["hot_technology"] = (
    technology_summary["hot_technology"].astype(int)
)

technology_summary["in_demand"] = (
    technology_summary["in_demand"].astype(int)
)

# Create a simple technology priority score
technology_summary["Technology_Priority"] = (
    technology_summary["hot_technology"] * 2
    + technology_summary["in_demand"] * 3
)

# Sort highest priority first
technology_summary = technology_summary.sort_values(
    by=["Technology_Priority", "in_demand", "hot_technology"],
    ascending=False
).reset_index(drop=True)

print(f"\nOriginal software-skill records: {len(analyst_software_skills)}")
print(f"Valid software records analyzed: {len(software_analysis)}")
print(f"Unique technologies identified: {len(technology_summary)}")

print("\nTOP TECHNOLOGIES")
print("=" * 70)

display(
    technology_summary.head(30)
)

print("\n✓ Software technology analysis completed successfully.")


STEP 22: ANALYZING SOFTWARE TECHNOLOGY DEMAND

Original software-skill records: 203
Valid software records analyzed: 203
Unique technologies identified: 203

TOP TECHNOLOGIES


,Workplace Example,software_category,hot_technology,in_demand,Technology_Priority
0,Amazon Web Services AWS software,Data base user interface and query software,1,1,5
1,Microsoft Azure software,Development environment software,1,1,5
2,Microsoft Excel,Spreadsheet software,1,1,5
3,Microsoft Office software,Office suite software,1,1,5
4,Microsoft Power BI,Business intelligence and data analysis software,1,1,5
5,Microsoft PowerPoint,Presentation software,1,1,5
6,Python,Object or component oriented development software,1,1,5
7,R,Object or component oriented development software,1,1,5
8,SAP software,Enterprise resource planning ERP software,1,1,5
9,SAS,Analytical or scientific software,1,1,5



✓ Software technology analysis completed successfully.


In [ ]:
print("\n" + "=" * 70)
print("STEP 22A: CLASSIFYING TECHNOLOGY DEMAND")
print("=" * 70)

technology_analysis = software_analysis.copy()

# Create demand classification
def classify_technology(row):
    hot = row["Hot Technology"] == "Y"
    demand = row["In Demand"] == "Y"

    if hot and demand:
        return "Hot + In Demand"
    elif hot:
        return "Hot Technology"
    elif demand:
        return "In Demand"
    else:
        return "Other"

technology_analysis["Demand_Category"] = (
    technology_analysis.apply(classify_technology, axis=1)
)

# Keep one row per technology
technology_summary = (
    technology_analysis[
        [
            "Workplace Example",
            "Element Name",
            "Hot Technology",
            "In Demand",
            "Demand_Category"
        ]
    ]
    .drop_duplicates(subset=["Workplace Example"])
    .sort_values(
        by=[
            "Demand_Category",
            "Workplace Example"
        ],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

# Display summary counts
print("\nDEMAND CATEGORY COUNTS")
print("=" * 70)

print(
    technology_summary["Demand_Category"]
    .value_counts()
)

print("\nTECHNOLOGY SUMMARY")
print("=" * 70)

display(technology_summary)

print("\n✓ Technology demand classification completed successfully.")


STEP 22A: CLASSIFYING TECHNOLOGY DEMAND

DEMAND CATEGORY COUNTS
Demand_Category
Other              103
Hot Technology      86
Hot + In Demand     14
Name: count, dtype: int64

TECHNOLOGY SUMMARY


,Workplace Example,Element Name,Hot Technology,In Demand,Demand_Category
0,Amazon Web Services AWS software,Data base user interface and query software,Y,Y,Hot + In Demand
1,Microsoft Azure software,Development environment software,Y,Y,Hot + In Demand
2,Microsoft Excel,Spreadsheet software,Y,Y,Hot + In Demand
3,Microsoft Office software,Office suite software,Y,Y,Hot + In Demand
4,Microsoft Power BI,Business intelligence and data analysis software,Y,Y,Hot + In Demand
...,...,...,...,...,...
198,Unified modeling language UML,Requirements analysis and system architecture ...,N,N,Other
199,Veritas NetBackup,Backup or archival software,N,N,Other
200,Virtual private networking VPN software,Network security and virtual private network V...,N,N,Other
201,Wireshark,Network monitoring software,N,N,Other



✓ Technology demand classification completed successfully.


In [ ]:
# ============================================================
# STEP 23: BUILDING O*NET RAG KNOWLEDGE BASE
# ============================================================

print("=" * 60)
print("STEP 23: BUILDING O*NET RAG KNOWLEDGE BASE")
print("=" * 60)

# Install required libraries
!pip -q install sentence-transformers faiss-cpu

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("\n✓ Required libraries ready.")

# ------------------------------------------------------------
# 1. PREPARE KNOWLEDGE DOCUMENTS
# ------------------------------------------------------------

rag_documents = []

# ---- Occupation information ----
occupation_row = occupation_data[
    occupation_data["O*NET-SOC Code"].astype(str) == str(TARGET_OCCUPATION_CODE)
]

if not occupation_row.empty:
    row = occupation_row.iloc[0]

    occupation_text = (
        f"Occupation: {row['Title']}. "
        f"Occupation Code: {row['O*NET-SOC Code']}. "
        f"Description: {row['Description']}"
    )

    rag_documents.append({
        "text": occupation_text,
        "doc_type": "occupation",
        "source": "O*NET"
    })

# ---- Task information ----
occupation_tasks = task_statements[
    task_statements["O*NET-SOC Code"].astype(str) == str(TARGET_OCCUPATION_CODE)
].copy()

for _, row in occupation_tasks.iterrows():

    task_text = (
        f"Occupation: {row['Title']}. "
        f"Task ID: {row['Task ID']}. "
        f"Task: {row['Task']}. "
        f"Task Type: {row['Task Type']}."
    )

    rag_documents.append({
        "text": task_text,
        "doc_type": "task",
        "source": "O*NET"
    })

# ---- Software skills ----
occupation_software = software_skills[
    software_skills["O*NET-SOC Code"].astype(str) == str(TARGET_OCCUPATION_CODE)
].copy()

for _, row in occupation_software.iterrows():

    software_text = (
        f"Occupation: {row['Title']}. "
        f"Software/Technology: {row['Workplace Example']}. "
        f"Element: {row['Element Name']}. "
        f"Hot Technology: {row['Hot Technology']}. "
        f"In Demand: {row['In Demand']}."
    )

    rag_documents.append({
        "text": software_text,
        "doc_type": "software",
        "source": "O*NET"
    })

# ------------------------------------------------------------
# 2. CREATE DATAFRAME
# ------------------------------------------------------------

rag_df = pd.DataFrame(rag_documents)

print("\nRAG DOCUMENT SUMMARY")
print("-" * 60)
print(rag_df["doc_type"].value_counts())

print(f"\nTotal RAG documents: {len(rag_df)}")

# Safety check
if rag_df.empty:
    raise ValueError("RAG knowledge base is empty.")

# ------------------------------------------------------------
# 3. CREATE EMBEDDINGS
# ------------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✓ Embedding model loaded.")

print("\nCreating embeddings...")

embeddings = embedding_model.encode(
    rag_df["text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print(f"✓ Embeddings created.")
print(f"Embedding shape: {embeddings.shape}")

# ------------------------------------------------------------
# 4. BUILD FAISS VECTOR INDEX
# ------------------------------------------------------------

embedding_dimension = embeddings.shape[1]

rag_index = faiss.IndexFlatIP(embedding_dimension)

rag_index.add(embeddings)

print("\n✓ FAISS index created.")
print(f"Vectors stored: {rag_index.ntotal}")

# ------------------------------------------------------------
# 5. RAG RETRIEVAL FUNCTION
# ------------------------------------------------------------

def retrieve_rag_context(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        min(top_k, len(rag_df))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = rag_df.iloc[idx].to_dict()
        result["similarity"] = float(score)

        results.append(result)

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 6. TEST RETRIEVAL
# ------------------------------------------------------------

test_query = (
    "Analyze business intelligence data, create reports, "
    "maintain BI tools and communicate insights."
)

retrieved_context = retrieve_rag_context(
    test_query,
    top_k=5
)

print("\n" + "=" * 60)
print("RAG RETRIEVAL TEST")
print("=" * 60)

display(
    retrieved_context[
        ["doc_type", "source", "similarity", "text"]
    ]
)

print("\n✓ RAG knowledge base built successfully.")
print("✓ FAISS retrieval tested successfully.")
print("✓ STEP 23 COMPLETED SUCCESSFULLY.")

STEP 23: BUILDING O*NET RAG KNOWLEDGE BASE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.2 MB/s eta 0:00:00

✓ Required libraries ready.

RAG DOCUMENT SUMMARY
------------------------------------------------------------
doc_type
software      203
task           17
occupation      1
Name: count, dtype: int64

Total RAG documents: 221

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

✓ Embedding model loaded.

Creating embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✓ Embeddings created.
Embedding shape: (221, 384)

✓ FAISS index created.
Vectors stored: 221

RAG RETRIEVAL TEST


,doc_type,source,similarity,text
0,software,O*NET,0.622285,Occupation: Business Intelligence Analysts. So...
1,software,O*NET,0.603308,Occupation: Business Intelligence Analysts. So...
2,occupation,O*NET,0.602998,Occupation: Business Intelligence Analysts. Oc...
3,task,O*NET,0.584197,Occupation: Business Intelligence Analysts. Ta...
4,task,O*NET,0.582804,Occupation: Business Intelligence Analysts. Ta...



✓ RAG knowledge base built successfully.
✓ FAISS retrieval tested successfully.
✓ STEP 23 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 24: TESTING RAG RETRIEVAL
# ============================================================

print("=" * 70)
print("STEP 24: TESTING RAG RETRIEVAL")
print("=" * 70)


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import faiss


# ============================================================
# 2. CHECK REQUIRED RAG COMPONENTS
# ============================================================

print("\nChecking RAG components...")

# Check embedding model
if "embedding_model" not in globals() or embedding_model is None:
    raise RuntimeError(
        "Embedding model is missing. "
        "Please run STEP 23 first."
    )

print("✓ Embedding model found.")


# Check RAG documents
if "rag_documents" not in globals() or rag_documents is None:
    raise RuntimeError(
        "rag_documents is missing. "
        "Please run STEP 23 first."
    )

if not isinstance(rag_documents, list):
    raise TypeError(
        f"rag_documents must be a list, "
        f"but found {type(rag_documents).__name__}."
    )

if len(rag_documents) == 0:
    raise ValueError(
        "rag_documents is empty. "
        "STEP 23 did not create any documents."
    )

print(f"✓ RAG documents found: {len(rag_documents)}")


# ============================================================
# 3. VALIDATE RAG DOCUMENT STRUCTURE
# ============================================================

required_doc_fields = ["text", "doc_type", "source"]

for i, doc in enumerate(rag_documents):

    if not isinstance(doc, dict):
        raise TypeError(
            f"RAG document {i} is not a dictionary."
        )

    if "text" not in doc:
        raise ValueError(
            f"RAG document {i} does not contain 'text'."
        )

    if not str(doc["text"]).strip():
        raise ValueError(
            f"RAG document {i} contains empty text."
        )

print("✓ RAG document structure validated.")


# ============================================================
# 4. CREATE DOCUMENT TEXT LIST
# ============================================================

rag_texts = [
    str(doc["text"]).strip()
    for doc in rag_documents
]

print(
    f"✓ Prepared {len(rag_texts)} document texts."
)


# ============================================================
# 5. CREATE / REBUILD FAISS INDEX
# ============================================================
#
# IMPORTANT:
# We intentionally rebuild the index here.
#
# This prevents:
#   - faiss_index being None
#   - stale indexes
#   - dimension mismatch
#   - undefined variable errors
#
# ============================================================

print("\nBuilding FAISS index...")

# Generate embeddings for all RAG documents
rag_embeddings = embedding_model.encode(
    rag_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Force float32 for FAISS
rag_embeddings = np.asarray(
    rag_embeddings,
    dtype="float32"
)


# Validate embedding shape
if rag_embeddings.ndim != 2:
    raise ValueError(
        "RAG embeddings must be a 2-dimensional array."
    )

if rag_embeddings.shape[0] != len(rag_documents):
    raise ValueError(
        f"Embedding/document count mismatch: "
        f"{rag_embeddings.shape[0]} embeddings "
        f"for {len(rag_documents)} documents."
    )


embedding_dimension = rag_embeddings.shape[1]

print(
    f"✓ Embeddings created: "
    f"{rag_embeddings.shape[0]} × {embedding_dimension}"
)


# Create a fresh FAISS inner-product index.
#
# Because embeddings are normalized,
# inner product = cosine similarity.
#
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(rag_embeddings)


# Validate index
if faiss_index is None:
    raise RuntimeError(
        "FAISS index creation failed."
    )

if faiss_index.ntotal != len(rag_documents):
    raise ValueError(
        f"FAISS/document count mismatch: "
        f"index contains {faiss_index.ntotal}, "
        f"documents contain {len(rag_documents)}."
    )

print(
    f"✓ FAISS index created successfully."
)

print(
    f"✓ Vectors stored: {faiss_index.ntotal}"
)


# ============================================================
# 6. DEFINE RAG RETRIEVAL FUNCTION
# ============================================================

def retrieve_rag_documents(
    query,
    top_k=5
):

    if not isinstance(query, str):
        raise TypeError(
            "Query must be a string."
        )

    query = query.strip()

    if not query:
        raise ValueError(
            "Query cannot be empty."
        )

    if faiss_index is None:
        raise RuntimeError(
            "FAISS index is not available."
        )

    if faiss_index.ntotal == 0:
        raise RuntimeError(
            "FAISS index contains no vectors."
        )

    # Do not request more documents than available
    top_k = min(
        int(top_k),
        faiss_index.ntotal
    )

    # Create query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Search FAISS
    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for rank, (idx, similarity) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):

        # FAISS may return -1 for missing results
        if idx < 0:
            continue

        idx = int(idx)

        if idx >= len(rag_documents):
            continue

        doc = rag_documents[idx]

        retrieved_documents.append({
            "rank": rank,
            "index": idx,
            "similarity": float(similarity),
            "doc_type": doc.get(
                "doc_type",
                "Unknown"
            ),
            "source": doc.get(
                "source",
                "Unknown"
            ),
            "text": str(
                doc.get(
                    "text",
                    ""
                )
            )
        })

    return retrieved_documents


# ============================================================
# 7. TEST QUERIES
# ============================================================

test_queries = [

    "What software technologies are important for Business Intelligence Analysts?",

    "What tasks do Business Intelligence Analysts perform?",

    "What technical skills are required for Business Intelligence Analysts?"

]


# ============================================================
# 8. RUN RAG RETRIEVAL TESTS
# ============================================================

all_retrieval_results = []


for query_number, query in enumerate(
    test_queries,
    start=1
):

    print("\n" + "-" * 70)
    print(f"QUERY {query_number}")
    print("-" * 70)

    print(
        f"Question: {query}"
    )

    try:

        results = retrieve_rag_documents(
            query,
            top_k=5
        )

        if not results:
            print(
                "⚠ No documents were retrieved."
            )
            continue

        print(
            "\nTOP RETRIEVED DOCUMENTS:"
        )

        for result in results:

            print(
                f"\n{result['rank']}. "
                f"Document Type: "
                f"{result['doc_type']}"
            )

            print(
                f"   Similarity: "
                f"{result['similarity']:.4f}"
            )

            print(
                f"   Source: "
                f"{result['source']}"
            )

            preview = result["text"][:300]

            print(
                f"   Text: "
                f"{preview}..."
            )

        all_retrieval_results.append({
            "query": query,
            "results": results
        })

    except Exception as e:

        print(
            f"\n❌ Retrieval failed for query "
            f"{query_number}"
        )

        print(
            f"{type(e).__name__}: {e}"
        )

        raise


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("RAG VALIDATION")
print("=" * 70)


if faiss_index is None:
    raise RuntimeError(
        "Final validation failed: FAISS index is None."
    )


if faiss_index.ntotal != len(rag_documents):
    raise ValueError(
        "Final validation failed: "
        "FAISS index size does not match "
        "RAG document count."
    )


if len(all_retrieval_results) != len(test_queries):
    raise ValueError(
        "Not all test queries returned results."
    )


for result in all_retrieval_results:

    if len(result["results"]) == 0:
        raise ValueError(
            f"No results returned for query: "
            f"{result['query']}"
        )


print(
    f"✓ Embedding model successfully used."
)

print(
    f"✓ RAG documents successfully validated: "
    f"{len(rag_documents)}"
)

print(
    f"✓ FAISS index successfully validated: "
    f"{faiss_index.ntotal} vectors"
)

print(
    f"✓ {len(all_retrieval_results)} test queries "
    f"successfully executed."
)

print(
    "✓ Retrieved documents mapped correctly "
    "back to RAG documents."
)

print("\n" + "=" * 70)
print("✓ STEP 24 COMPLETED SUCCESSFULLY.")
print("=" * 70)

STEP 24: TESTING RAG RETRIEVAL

Checking RAG components...
✓ Embedding model found.
✓ RAG documents found: 221
✓ RAG document structure validated.
✓ Prepared 221 document texts.

Building FAISS index...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✓ Embeddings created: 221 × 384
✓ FAISS index created successfully.
✓ Vectors stored: 221

----------------------------------------------------------------------
QUERY 1
----------------------------------------------------------------------
Question: What software technologies are important for Business Intelligence Analysts?

TOP RETRIEVED DOCUMENTS:

1. Document Type: software
   Similarity: 0.6357
   Source: O*NET
   Text: Occupation: Business Intelligence Analysts. Software/Technology: Microsoft Power BI. Element: Business intelligence and data analysis software. Hot Technology: Y. In Demand: Y....

2. Document Type: software
   Similarity: 0.6292
   Source: O*NET
   Text: Occupation: Business Intelligence Analysts. Software/Technology: Extensible markup language XML. Element: Enterprise application integration software. Hot Technology: Y. In Demand: N....

3. Document Type: software
   Similarity: 0.6263
   Source: O*NET
   Text: Occupation: Business Intelligence Analysts. Softwar

In [ ]:
# ================================================================
# STEP 25: EXTRACT AI-EXPOSURE SIGNALS FOR EACH ANALYST TASK
# ================================================================

print("=" * 70)
print("STEP 25: EXTRACTING AI-EXPOSURE SIGNALS")
print("=" * 70)

import json
import re
import time
import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------

required_variables = [
    "task_level_metrics",
    "rag_documents",
    "faiss_index",
    "embedding_model"
]

missing = [v for v in required_variables if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required variables: {missing}. "
        "Run the previous pipeline steps first."
    )

print(f"✓ Task metrics found: {len(task_level_metrics)}")
print(f"✓ RAG documents found: {len(rag_documents)}")
print("✓ FAISS index found.")
print("✓ Embedding model found.")


# ------------------------------------------------
# 2. FIND THE LLM CLIENT
# ------------------------------------------------
#
# This supports the common clients used in this project.
# If your notebook already created one, it will reuse it.
#

llm_client = None
llm_provider = None

# OpenAI-style client
if "client" in globals():
    try:
        if hasattr(client, "chat"):
            llm_client = client
            llm_provider = "openai_style"
    except Exception:
        pass

# Groq client
if llm_client is None and "groq_client" in globals():
    try:
        if hasattr(groq_client, "chat"):
            llm_client = groq_client
            llm_provider = "groq"
    except Exception:
        pass

# Another common variable name
if llm_client is None and "llm" in globals():
    try:
        if hasattr(llm, "invoke"):
            llm_client = llm
            llm_provider = "langchain"
    except Exception:
        pass


if llm_client is None:
    raise RuntimeError(
        "No LLM client found. Create your Groq/OpenAI client before "
        "running Step 25."
    )

print(f"✓ LLM client found: {llm_provider}")


# ------------------------------------------------
# 3. HELPER: GET TASK TEXT
# ------------------------------------------------

def get_task_text(row):
    """
    Safely extract the task description regardless of
    minor dataframe column differences.
    """

    possible_columns = [
        "Task",
        "task",
        "Task Description",
        "task_description"
    ]

    for col in possible_columns:
        if col in row.index:
            value = row[col]

            if pd.notna(value):
                return str(value).strip()

    return ""


# ------------------------------------------------
# 4. HELPER: RETRIEVE RAG EVIDENCE
# ------------------------------------------------

def retrieve_task_evidence(task_text, k=5):

    if not task_text:
        return []

    query_embedding = embedding_model.encode(
        [task_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_embedding,
        k
    )

    evidence = []

    for idx, distance in zip(indices[0], distances[0]):

        if idx < 0 or idx >= len(rag_documents):
            continue

        doc = rag_documents[idx]

        evidence.append({
            "doc_type": doc.get("doc_type", "unknown"),
            "source": doc.get("source", "unknown"),
            "text": str(doc.get("text", "")),
            "similarity": float(distance)
        })

    return evidence


# ------------------------------------------------
# 5. HELPER: CALL LLM
# ------------------------------------------------

def call_llm(prompt):

    # --------------------------------------------
    # LangChain
    # --------------------------------------------
    if llm_provider == "langchain":

        response = llm_client.invoke(prompt)

        if hasattr(response, "content"):
            return response.content

        return str(response)

    # --------------------------------------------
    # OpenAI / Groq compatible clients
    # --------------------------------------------
    response = llm_client.chat.completions.create(

        model=(
            globals().get("LLM_MODEL")
            or globals().get("MODEL_NAME")
            or "llama-3.1-8b-instant"
        ),

        messages=[
            {
                "role": "system",
                "content": (
                    "You are an occupational AI-exposure analyst. "
                    "Use only the supplied evidence and task description. "
                    "Return valid JSON only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


# ------------------------------------------------
# 6. PARSE JSON SAFELY
# ------------------------------------------------

def parse_signal_json(raw):

    raw = str(raw).strip()

    # Remove markdown code fences
    raw = re.sub(
        r"```(?:json)?",
        "",
        raw,
        flags=re.IGNORECASE
    )

    raw = raw.replace("```", "").strip()

    # Try direct JSON
    try:
        return json.loads(raw)
    except Exception:
        pass

    # Try extracting JSON object
    match = re.search(
        r"\{.*\}",
        raw,
        flags=re.DOTALL
    )

    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    raise ValueError(
        f"Could not parse LLM JSON:\n{raw[:1000]}"
    )


# ------------------------------------------------
# 7. EXTRACT SIGNALS FOR ONE TASK
# ------------------------------------------------

def analyze_task(task_text, evidence):

    evidence_text = "\n\n".join(
        [
            (
                f"[Source: {e['source']} | "
                f"Type: {e['doc_type']} | "
                f"Similarity: {e['similarity']:.4f}]\n"
                f"{e['text'][:1200]}"
            )
            for e in evidence
        ]
    )

    prompt = f"""
Analyze this occupational task for AI transformation.

TASK:
{task_text}

RETRIEVED EVIDENCE:
{evidence_text}

Evaluate exactly these three signals.

1. tool_coverage
Question:
Can current AI/software tools already perform the output of this
task today?

Score:
1 = very low current tool capability
2 = low
3 = moderate
4 = high
5 = very high

2. trend_momentum
Question:
Is this type of task moving toward greater AI automation over time?

Score:
1 = little/no evidence of movement
2 = low
3 = moderate
4 = strong
5 = very strong

3. judgment_reliance
Question:
How much does the task require contextual judgment, accountability,
human decision-making, or real-time/physical presence?

Score:
1 = very little human judgment required
2 = low
3 = moderate
4 = high
5 = very high

IMPORTANT:
- Do NOT calculate the final exposure score yourself.
- Do NOT make up evidence.
- Base the reasoning on the supplied task and retrieved evidence.
- If evidence is weak, use a conservative score.
- Return ONLY valid JSON.

Required JSON:

{{
    "tool_coverage": 1,
    "trend_momentum": 1,
    "judgment_reliance": 1,
    "rationale": "Short explanation grounded in the evidence.",
    "evidence_sources": ["source1", "source2"]
}}
"""

    raw = call_llm(prompt)

    result = parse_signal_json(raw)

    # --------------------------------------------
    # Validate signal values
    # --------------------------------------------

    signal_names = [
        "tool_coverage",
        "trend_momentum",
        "judgment_reliance"
    ]

    for signal in signal_names:

        if signal not in result:
            raise ValueError(
                f"LLM response missing {signal}"
            )

        result[signal] = float(result[signal])

        # Clamp to allowed 1-5 range
        result[signal] = max(
            1.0,
            min(5.0, result[signal])
        )

    result["rationale"] = str(
        result.get(
            "rationale",
            "No rationale provided."
        )
    )

    result["evidence_sources"] = result.get(
        "evidence_sources",
        []
    )

    return result


# ------------------------------------------------
# 8. PROCESS ALL 17 TASKS
# ------------------------------------------------

signal_results = []

print("\nProcessing analyst tasks...")
print("-" * 70)

for i, (_, row) in enumerate(
    task_level_metrics.iterrows(),
    start=1
):

    task_id = row.get(
        "Task ID",
        row.get("task_id", i)
    )

    task_text = get_task_text(row)

    print(
        f"[{i:02d}/{len(task_level_metrics)}] "
        f"Task ID {task_id}: "
        f"{task_text[:70]}..."
    )

    try:

        # Retrieve grounding evidence
        evidence = retrieve_task_evidence(
            task_text,
            k=5
        )

        # Extract signals
        signals = analyze_task(
            task_text,
            evidence
        )

        signal_results.append({
            "Task ID": task_id,
            "Task": task_text,
            "tool_coverage": signals["tool_coverage"],
            "trend_momentum": signals["trend_momentum"],
            "judgment_reliance": signals["judgment_reliance"],
            "rationale": signals["rationale"],
            "evidence_sources": signals["evidence_sources"]
        })

        print(
            f"    ✓ Coverage={signals['tool_coverage']:.1f} | "
            f"Trend={signals['trend_momentum']:.1f} | "
            f"Judgment={signals['judgment_reliance']:.1f}"
        )

    except Exception as e:

        print(
            f"    ⚠ Failed: {str(e)[:150]}"
        )

        # Conservative fallback rather than crashing entire pipeline
        signal_results.append({
            "Task ID": task_id,
            "Task": task_text,
            "tool_coverage": 3.0,
            "trend_momentum": 3.0,
            "judgment_reliance": 3.0,
            "rationale": (
                "Conservative fallback because signal extraction "
                "failed."
            ),
            "evidence_sources": []
        })

    # Small delay to avoid rate-limit bursts
    time.sleep(0.2)


# ------------------------------------------------
# 9. CREATE SIGNAL DATAFRAME
# ------------------------------------------------

task_signals = pd.DataFrame(signal_results)

print("\n" + "=" * 70)
print("TASK SIGNALS")
print("=" * 70)

display(task_signals)


# ------------------------------------------------
# 10. VALIDATION
# ------------------------------------------------

expected_tasks = len(task_level_metrics)

if len(task_signals) != expected_tasks:
    raise ValueError(
        f"Expected {expected_tasks} task signal records, "
        f"but found {len(task_signals)}."
    )

for column in [
    "tool_coverage",
    "trend_momentum",
    "judgment_reliance"
]:

    if task_signals[column].isna().any():
        raise ValueError(
            f"{column} contains missing values."
        )

    if not task_signals[column].between(1, 5).all():
        raise ValueError(
            f"{column} contains values outside 1-5."
        )


print("\n✓ All task signals extracted.")
print("✓ All signal values are within 1-5.")
print(f"✓ Exactly {len(task_signals)} tasks analyzed.")
print("✓ STEP 25 COMPLETED SUCCESSFULLY.")

STEP 25: EXTRACTING AI-EXPOSURE SIGNALS
✓ Task metrics found: 17
✓ RAG documents found: 221
✓ FAISS index found.
✓ Embedding model found.
✓ LLM client found: langchain

Processing analyst tasks...
----------------------------------------------------------------------
[01/17] Task ID 16150: Generate standard or custom reports summarizing business, financial, o...
    ✓ Coverage=4.0 | Trend=4.0 | Judgment=4.0
[02/17] Task ID 16139: Maintain or update business intelligence tools, databases, dashboards,...
    ✓ Coverage=3.0 | Trend=4.0 | Judgment=4.0
[03/17] Task ID 16140: Manage timely flow of business intelligence information to users....
    ✓ Coverage=3.0 | Trend=3.0 | Judgment=4.0
[04/17] Task ID 16134: Provide technical support for existing reports, dashboards, or other t...
    ✓ Coverage=2.0 | Trend=2.0 | Judgment=4.0
[05/17] Task ID 16141: Identify and analyze industry or geographic trends with business strat...
    ✓ Coverage=3.0 | Trend=3.0 | Judgment=4.0
[06/17] Task ID 16142:

,Task ID,Task,tool_coverage,trend_momentum,judgment_reliance,rationale,evidence_sources
0,16150,Generate standard or custom reports summarizin...,4.0,4.0,4.0,The task is a core activity for Business Intel...,"[source1, source2]"
1,16139,Maintain or update business intelligence tools...,3.0,4.0,4.0,"The task involves using BI software (Power BI,...","[source1, source2]"
2,16140,Manage timely flow of business intelligence in...,3.0,3.0,4.0,The task is listed as a core responsibility fo...,"[source1, source2]"
3,16134,Provide technical support for existing reports...,2.0,2.0,4.0,"The evidence lists reporting platforms (SSRS, ...","[source1, source2, source3, source4]"
4,16141,Identify and analyze industry or geographic tr...,3.0,3.0,4.0,The task is a core responsibility of Business ...,"[source1, source2]"
5,16142,Document specifications for business intellige...,3.0,4.0,4.0,Current AI tools can draft specification docum...,"[O*NET-16142, O*NET-16137]"
6,16144,"Create business intelligence tools or systems,...",3.0,4.0,4.0,"Current BI software (e.g., Power BI, SQL, Tran...","[O*NET task 16144, O*NET software Structured q..."
7,16146,Collect business intelligence data from availa...,2.0,2.0,4.0,The O*NET task description shows that Business...,"[source1, source2]"
8,16143,"Disseminate information regarding tools, repor...",3.0,4.0,4.0,The task is a core activity for Business Intel...,"[source3, source1]"
9,16145,Conduct or coordinate tests to ensure that int...,2.0,3.0,4.0,The O*NET evidence shows Business Intelligence...,"[source1, source2]"



✓ All task signals extracted.
✓ All signal values are within 1-5.
✓ Exactly 17 tasks analyzed.
✓ STEP 25 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 26: CALCULATING AI EXPOSURE & TASK CLASSIFICATION
# ============================================================

print("=" * 70)
print("STEP 26: CALCULATING AI EXPOSURE & TASK CLASSIFICATION")
print("=" * 70)

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Validate input
# ------------------------------------------------------------

if "task_signals" not in globals():
    raise RuntimeError(
        "task_signals not found. Please run STEP 25 first."
    )

if "ranked_tasks" not in globals():
    raise RuntimeError(
        "ranked_tasks not found. Please run STEP 20 first."
    )

print(f"\nTask signal records: {len(task_signals)}")
print(f"Ranked task records: {len(ranked_tasks)}")


# ------------------------------------------------------------
# 2. Merge task information + AI signals
# ------------------------------------------------------------

analysis_df = ranked_tasks[
    [
        "Task ID",
        "Task",
        "Importance",
        "Relevance",
        "Frequency",
        "Task_Priority"
    ]
].merge(
    task_signals[
        [
            "Task ID",
            "tool_coverage",
            "trend_momentum",
            "judgment_reliance",
            "rationale",
            "evidence_sources"
        ]
    ],
    on="Task ID",
    how="left"
)


# ------------------------------------------------------------
# 3. Validate merge
# ------------------------------------------------------------

if len(analysis_df) != len(ranked_tasks):
    raise ValueError(
        "Task merge failed. The number of tasks changed."
    )

if analysis_df[
    [
        "tool_coverage",
        "trend_momentum",
        "judgment_reliance"
    ]
].isna().any().any():

    raise ValueError(
        "Some tasks are missing AI-exposure signals."
    )

print("✓ Task and signal data merged successfully.")


# ------------------------------------------------------------
# 4. Detect fallback signal records
# ------------------------------------------------------------

analysis_df["Signal_Status"] = np.where(
    analysis_df["rationale"]
    .astype(str)
    .str.contains(
        "Conservative fallback",
        case=False,
        na=False
    ),
    "Fallback",
    "LLM"
)


# ------------------------------------------------------------
# 5. AUTOMATION POTENTIAL
# ------------------------------------------------------------

analysis_df["Automation_Potential"] = (
    analysis_df["tool_coverage"]
)


# ------------------------------------------------------------
# 6. AI EXPOSURE SCORE
#
# Formula from project design:
#
# 100 * (
#     0.5 * tool_coverage
#     + 0.3 * trend_momentum
#     + 0.2 * (6 - judgment_reliance)
# ) / 5
#
# ------------------------------------------------------------

analysis_df["Exposure_Score"] = (
    100
    * (
        0.5 * analysis_df["tool_coverage"]
        + 0.3 * analysis_df["trend_momentum"]
        + 0.2 * (
            6 - analysis_df["judgment_reliance"]
        )
    )
    / 5
)


# ------------------------------------------------------------
# 7. HUMAN DEPENDENCY
# ------------------------------------------------------------

analysis_df["Human_Dependency"] = (
    analysis_df["judgment_reliance"]
)


# ------------------------------------------------------------
# 8. CLASSIFICATION
# ------------------------------------------------------------

def classify_task(row):

    automation = row["Automation_Potential"]
    human = row["Human_Dependency"]

    if automation >= 4 and human <= 2:
        return "Highly Automatable"

    elif human >= 4:
        return "Human-dependent"

    else:
        return "AI-augmented"


analysis_df["Classification"] = (
    analysis_df.apply(
        classify_task,
        axis=1
    )
)


# ------------------------------------------------------------
# 9. ROUND SCORES
# ------------------------------------------------------------

analysis_df["Automation_Potential"] = (
    analysis_df["Automation_Potential"].round(2)
)

analysis_df["Exposure_Score"] = (
    analysis_df["Exposure_Score"].round(2)
)

analysis_df["Human_Dependency"] = (
    analysis_df["Human_Dependency"].round(2)
)


# ------------------------------------------------------------
# 10. SORT BY EXPOSURE
# ------------------------------------------------------------

analysis_df = (
    analysis_df
    .sort_values(
        by="Exposure_Score",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. TASK RANK
# ------------------------------------------------------------

analysis_df["Exposure_Rank"] = (
    analysis_df.index + 1
)


# ------------------------------------------------------------
# 12. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK AI-EXPOSURE ANALYSIS")
print("=" * 70)

display(
    analysis_df[
        [
            "Exposure_Rank",
            "Task ID",
            "Task",
            "tool_coverage",
            "trend_momentum",
            "judgment_reliance",
            "Automation_Potential",
            "Exposure_Score",
            "Human_Dependency",
            "Classification",
            "Signal_Status"
        ]
    ]
)


# ------------------------------------------------------------
# 13. OVERALL JOB TRANSFORMATION SCORE
#
# Weighted by task importance.
# ------------------------------------------------------------

total_importance = (
    analysis_df["Importance"].sum()
)

if total_importance > 0:

    overall_transformation_score = (
        (
            analysis_df["Exposure_Score"]
            * analysis_df["Importance"]
        ).sum()
        / total_importance
    )

else:

    overall_transformation_score = (
        analysis_df["Exposure_Score"].mean()
    )


overall_transformation_score = round(
    float(overall_transformation_score),
    2
)


# ------------------------------------------------------------
# 14. CLASSIFICATION DISTRIBUTION
# ------------------------------------------------------------

classification_counts = (
    analysis_df["Classification"]
    .value_counts()
)


# ------------------------------------------------------------
# 15. FINAL VALIDATION
# ------------------------------------------------------------

if len(analysis_df) != 17:
    raise ValueError(
        f"Expected 17 tasks, found {len(analysis_df)}."
    )

if not analysis_df["Exposure_Score"].between(
    0, 100
).all():

    raise ValueError(
        "Exposure score contains values outside 0–100."
    )

if not analysis_df[
    "Automation_Potential"
].between(1, 5).all():

    raise ValueError(
        "Automation Potential contains invalid values."
    )

if not analysis_df[
    "Human_Dependency"
].between(1, 5).all():

    raise ValueError(
        "Human Dependency contains invalid values."
    )


# ------------------------------------------------------------
# 16. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("JOB-LEVEL SUMMARY")
print("=" * 70)

print(
    f"\nOccupation: {TARGET_OCCUPATION_TITLE}"
)

print(
    f"O*NET-SOC Code: {TARGET_OCCUPATION_CODE}"
)

print(
    f"Overall AI Exposure Score: "
    f"{overall_transformation_score}/100"
)

print("\nTask Classification Distribution:")

for category, count in classification_counts.items():

    print(
        f"- {category}: {count}"
    )


fallback_count = (
    analysis_df["Signal_Status"]
    .eq("Fallback")
    .sum()
)

print(
    f"\nLLM-generated task analyses: "
    f"{len(analysis_df) - fallback_count}"
)

print(
    f"Fallback analyses: {fallback_count}"
)


print("\n✓ AI exposure scores calculated.")
print("✓ Automation potential calculated.")
print("✓ Human dependency calculated.")
print("✓ Tasks classified successfully.")
print("✓ Overall job transformation score calculated.")
print("✓ STEP 26 COMPLETED SUCCESSFULLY.")

STEP 26: CALCULATING AI EXPOSURE & TASK CLASSIFICATION

Task signal records: 17
Ranked task records: 17
✓ Task and signal data merged successfully.

TASK AI-EXPOSURE ANALYSIS


,Exposure_Rank,Task ID,Task,tool_coverage,trend_momentum,judgment_reliance,Automation_Potential,Exposure_Score,Human_Dependency,Classification,Signal_Status
0,1,16150,Generate standard or custom reports summarizin...,4.0,4.0,4.0,4.0,72.0,4.0,Human-dependent,LLM
1,2,16136,Identify or monitor current and potential cust...,4.0,4.0,4.0,4.0,72.0,4.0,Human-dependent,LLM
2,3,16135,"Maintain library of model documents, templates...",4.0,3.0,3.0,4.0,70.0,3.0,AI-augmented,LLM
3,4,16139,Maintain or update business intelligence tools...,3.0,4.0,4.0,3.0,62.0,4.0,Human-dependent,LLM
4,5,16142,Document specifications for business intellige...,3.0,4.0,4.0,3.0,62.0,4.0,Human-dependent,LLM
5,6,16144,"Create business intelligence tools or systems,...",3.0,4.0,4.0,3.0,62.0,4.0,Human-dependent,LLM
6,7,16143,"Disseminate information regarding tools, repor...",3.0,4.0,4.0,3.0,62.0,4.0,Human-dependent,LLM
7,8,16149,Synthesize current business intelligence or tr...,3.0,4.0,4.0,3.0,62.0,4.0,Human-dependent,LLM
8,9,16140,Manage timely flow of business intelligence in...,3.0,3.0,4.0,3.0,56.0,4.0,Human-dependent,LLM
9,10,16148,Analyze competitive market strategies through ...,3.0,3.0,4.0,3.0,56.0,4.0,Human-dependent,LLM



JOB-LEVEL SUMMARY

Occupation: Business Intelligence Analysts
O*NET-SOC Code: 15-2051.01
Overall AI Exposure Score: 56.52/100

Task Classification Distribution:
- Human-dependent: 16
- AI-augmented: 1

LLM-generated task analyses: 17
Fallback analyses: 0

✓ AI exposure scores calculated.
✓ Automation potential calculated.
✓ Human dependency calculated.
✓ Tasks classified successfully.
✓ Overall job transformation score calculated.
✓ STEP 26 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 27: SYNTHESIZING PRIORITY SKILLS
# ============================================================

print("=" * 70)
print("STEP 27: SYNTHESIZING PRIORITY SKILLS")
print("=" * 70)

import json
import re
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. VALIDATE REQUIRED DATA
# ------------------------------------------------------------

required_vars = [
    "analysis_df",
    "technology_summary"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Missing required variables: {missing_vars}. "
        "Please run the previous steps first."
    )

print("✓ Task analysis found.")
print("✓ Technology summary found.")


# ------------------------------------------------------------
# 2. IDENTIFY HIGH-EXPOSURE TASKS
# ------------------------------------------------------------

high_exposure_tasks = (
    analysis_df[
        analysis_df["Exposure_Score"] >= 60
    ][
        [
            "Task ID",
            "Task",
            "Exposure_Score",
            "Classification"
        ]
    ]
    .sort_values(
        "Exposure_Score",
        ascending=False
    )
)

print(
    f"\nHigh-exposure tasks: "
    f"{len(high_exposure_tasks)}"
)


# ------------------------------------------------------------
# 3. SELECT O*NET TECHNOLOGIES WITH STRONG DEMAND SIGNALS
# ------------------------------------------------------------

priority_technologies = (
    technology_summary[
        technology_summary["Demand_Category"].isin(
            [
                "Hot + In Demand",
                "Hot Technology",
                "In Demand"
            ]
        )
    ][
        [
            "Workplace Example",
            "Element Name",
            "Demand_Category"
        ]
    ]
    .head(40)
)

print(
    f"Priority technologies available: "
    f"{len(priority_technologies)}"
)


# ------------------------------------------------------------
# 4. BUILD COMPACT CONTEXT FOR LLM
# ------------------------------------------------------------

task_context = "\n".join(
    [
        f"- {row['Task']} "
        f"(Exposure: {row['Exposure_Score']}, "
        f"Classification: {row['Classification']})"
        for _, row in high_exposure_tasks.iterrows()
    ]
)

technology_context = "\n".join(
    [
        f"- {row['Workplace Example']} "
        f"({row['Element Name']}; "
        f"{row['Demand_Category']})"
        for _, row in priority_technologies.iterrows()
    ]
)


# ------------------------------------------------------------
# 5. MAKE SURE LLM CLIENT EXISTS
# ------------------------------------------------------------

if "llm_client" not in globals() or llm_client is None:
    raise RuntimeError(
        "LLM client not available. "
        "Please run STEP 25 first."
    )


# ------------------------------------------------------------
# 6. CREATE SKILL SYNTHESIS PROMPT
# ------------------------------------------------------------

skill_prompt = f"""
You are analyzing the occupation:

{TARGET_OCCUPATION_TITLE}

O*NET-SOC Code:
{TARGET_OCCUPATION_CODE}

The following are higher-AI-exposure tasks identified by the
project's task analysis:

{task_context}

The following technologies have O*NET demand signals for this
occupation:

{technology_context}

Synthesize the most useful skills for someone preparing for this
occupation.

Requirements:

1. Return between 6 and 10 skills.
2. Prefer concrete skills rather than vague traits.
3. Use skills supported by the tasks and technologies supplied.
4. Distinguish technical skills from human/business skills.
5. Do not claim that a skill is "future-proof".
6. Do not invent certifications or job requirements.
7. Return ONLY valid JSON.

Required format:

{{
    "skills": [
        {{
            "skill": "SQL",
            "category": "Technical",
            "priority": "High",
            "reason": "..."
        }}
    ]
}}
"""


# ------------------------------------------------------------
# 7. CALL LLM
# ------------------------------------------------------------

try:

    if llm_provider == "langchain":

        response = llm_client.invoke(
            skill_prompt
        )

        if hasattr(response, "content"):
            raw_response = response.content
        else:
            raw_response = str(response)

    else:

        model_name = (
            globals().get("LLM_MODEL")
            or globals().get("MODEL_NAME")
            or "llama-3.1-8b-instant"
        )

        response = llm_client.chat.completions.create(

            model=model_name,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a careful occupational skills "
                        "analyst. Return valid JSON only."
                    )
                },
                {
                    "role": "user",
                    "content": skill_prompt
                }
            ],

            temperature=0
        )

        raw_response = (
            response
            .choices[0]
            .message
            .content
        )


except Exception as e:

    raise RuntimeError(
        f"Skill synthesis LLM call failed: {e}"
    )


# ------------------------------------------------------------
# 8. PARSE JSON
# ------------------------------------------------------------

raw_response = str(raw_response).strip()

raw_response = re.sub(
    r"```(?:json)?",
    "",
    raw_response,
    flags=re.IGNORECASE
)

raw_response = raw_response.replace(
    "```",
    ""
).strip()


try:

    skill_result = json.loads(
        raw_response
    )

except Exception:

    match = re.search(
        r"\{.*\}",
        raw_response,
        flags=re.DOTALL
    )

    if not match:
        raise ValueError(
            "Could not extract JSON from LLM response."
        )

    skill_result = json.loads(
        match.group(0)
    )


# ------------------------------------------------------------
# 9. VALIDATE SKILLS
# ------------------------------------------------------------

if "skills" not in skill_result:
    raise ValueError(
        "LLM response does not contain 'skills'."
    )

skills = skill_result["skills"]

if not isinstance(skills, list):
    raise TypeError(
        "'skills' must be a list."
    )


# Keep only complete skill records
clean_skills = []

for item in skills:

    if not isinstance(item, dict):
        continue

    skill_name = str(
        item.get("skill", "")
    ).strip()

    category = str(
        item.get("category", "Technical")
    ).strip()

    priority = str(
        item.get("priority", "Medium")
    ).strip()

    reason = str(
        item.get("reason", "")
    ).strip()

    if not skill_name:
        continue

    clean_skills.append({
        "skill": skill_name,
        "category": category,
        "priority": priority,
        "reason": reason
    })


# Remove duplicate skill names
seen = set()
final_skills = []

for item in clean_skills:

    key = item["skill"].lower()

    if key in seen:
        continue

    seen.add(key)
    final_skills.append(item)


# Keep maximum 10
final_skills = final_skills[:10]


# ------------------------------------------------------------
# 10. CREATE DATAFRAME
# ------------------------------------------------------------

required_skills_df = pd.DataFrame(
    final_skills
)


# ------------------------------------------------------------
# 11. VALIDATE RESULT
# ------------------------------------------------------------

if len(required_skills_df) < 6:
    raise ValueError(
        f"Expected at least 6 synthesized skills, "
        f"but received {len(required_skills_df)}."
    )

if len(required_skills_df) > 10:
    required_skills_df = (
        required_skills_df.head(10)
        .copy()
    )


# ------------------------------------------------------------
# 12. DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRIORITY SKILLS FOR THE OCCUPATION")
print("=" * 70)

display(
    required_skills_df
)


print("\n✓ Skills synthesized from task and technology evidence.")
print(
    f"✓ {len(required_skills_df)} priority skills identified."
)
print("✓ STEP 27 COMPLETED SUCCESSFULLY.")

STEP 27: SYNTHESIZING PRIORITY SKILLS
✓ Task analysis found.
✓ Technology summary found.

High-exposure tasks: 8
Priority technologies available: 40

PRIORITY SKILLS FOR THE OCCUPATION


,skill,category,priority,reason
0,SQL (Structured Query Language),Technical,High,"Core for querying, updating, and managing the ..."
1,Python (including data‑analysis libraries such...,Technical,High,"Enables data extraction, transformation, and a..."
2,Data Visualization with Power BI and Tableau,Technical,High,Creates interactive dashboards and visual repo...
3,"Cloud Data‑Warehouse Platforms (AWS Redshift, ...",Technical,Medium,"Supports storage, scaling, and fast querying o..."
4,Statistical Analysis with R or SAS,Technical,Medium,Provides the statistical modeling and hypothes...
5,Business Requirements Documentation,Business,High,"Documenting specifications for reports, dashbo..."
6,Stakeholder Communication & Presentation (Powe...,Business,High,"Translates analytical findings into clear, act..."
7,Knowledge Management (maintaining model librar...,Business,Medium,"Ensures consistency, reusability, and rapid de..."



✓ Skills synthesized from task and technology evidence.
✓ 8 priority skills identified.
✓ STEP 27 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 28: SKILL GAP ANALYSIS
# ============================================================

print("=" * 70)
print("STEP 28: SKILL GAP ANALYSIS")
print("=" * 70)

import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CHECK REQUIRED DATA
# ------------------------------------------------------------

if "required_skills_df" not in globals():
    raise RuntimeError(
        "required_skills_df not found. Please run STEP 27 first."
    )

if "embedding_model" not in globals():
    raise RuntimeError(
        "embedding_model not found. Please run STEP 23 first."
    )

if required_skills_df.empty:
    raise ValueError(
        "required_skills_df is empty."
    )

if "skill" not in required_skills_df.columns:
    raise KeyError(
        "required_skills_df does not contain a 'skill' column."
    )

print(
    f"✓ Required skills found: {len(required_skills_df)}"
)

print("✓ Embedding model found.")


# ------------------------------------------------------------
# 2. USER SKILLS
# ------------------------------------------------------------
#
# Replace this example with YOUR actual skills.
#
# Keep them comma-separated.
#
# Example:
# "Python, Excel, SQL, Power BI, Statistics"
#

user_skills_text = (
    "Python, Excel, SQL, Statistics"
)

print("\nUSER SKILLS")
print("-" * 70)
print(user_skills_text)


# ------------------------------------------------------------
# 3. CLEAN USER SKILLS
# ------------------------------------------------------------

user_skills = [
    skill.strip()
    for skill in user_skills_text.split(",")
    if skill.strip()
]

# Remove duplicates
user_skills = list(
    dict.fromkeys(
        skill.lower() for skill in user_skills
    )
)

if not user_skills:
    raise ValueError(
        "No user skills were provided."
    )

print(
    f"\n✓ User skills detected: {len(user_skills)}"
)


# ------------------------------------------------------------
# 4. CLEAN REQUIRED SKILLS
# ------------------------------------------------------------

required_skills = (
    required_skills_df["skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

# Remove empty values
required_skills = [
    skill
    for skill in required_skills
    if skill
]

if not required_skills:
    raise ValueError(
        "No required skills were found."
    )


# ------------------------------------------------------------
# 5. CREATE EMBEDDINGS
# ------------------------------------------------------------

print("\nCreating skill embeddings...")

required_embeddings = embedding_model.encode(
    required_skills,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

user_embeddings = embedding_model.encode(
    user_skills,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

print("✓ Required skill embeddings created.")
print("✓ User skill embeddings created.")


# ------------------------------------------------------------
# 6. CALCULATE BEST COSINE SIMILARITY
# ------------------------------------------------------------
#
# Because embeddings are normalized,
# dot product = cosine similarity.
# ------------------------------------------------------------

skill_gap_results = []

for i, required_skill in enumerate(
    required_skills
):

    similarities = np.dot(
        user_embeddings,
        required_embeddings[i]
    )

    best_match_index = int(
        np.argmax(similarities)
    )

    best_similarity = float(
        similarities[best_match_index]
    )

    best_user_skill = user_skills[
        best_match_index
    ]


    # --------------------------------------------------------
    # Classification thresholds from project design
    # --------------------------------------------------------

    if best_similarity >= 0.75:

        status = "Have"

    elif best_similarity >= 0.50:

        status = "Partial"

    else:

        status = "Missing"


    skill_gap_results.append({

        "Required Skill": required_skill,

        "Best Matching User Skill":
            best_user_skill,

        "Similarity":
            round(best_similarity, 3),

        "Status":
            status
    })


# ------------------------------------------------------------
# 7. CREATE DATAFRAME
# ------------------------------------------------------------

skill_gap_df = pd.DataFrame(
    skill_gap_results
)


# ------------------------------------------------------------
# 8. ADD REQUIRED-SKILL METADATA
# ------------------------------------------------------------

metadata_columns = [
    "skill",
    "category",
    "priority"
]

available_metadata = [
    col
    for col in metadata_columns
    if col in required_skills_df.columns
]

if available_metadata:

    skill_gap_df = skill_gap_df.merge(
        required_skills_df[
            available_metadata
        ],
        left_on="Required Skill",
        right_on="skill",
        how="left"
    )

    skill_gap_df.drop(
        columns=["skill"],
        inplace=True,
        errors="ignore"
    )


# ------------------------------------------------------------
# 9. ORDER COLUMNS
# ------------------------------------------------------------

preferred_columns = [
    "Required Skill",
    "category",
    "priority",
    "Best Matching User Skill",
    "Similarity",
    "Status"
]

final_columns = [
    col
    for col in preferred_columns
    if col in skill_gap_df.columns
]

skill_gap_df = skill_gap_df[
    final_columns
]


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SKILL GAP RESULTS")
print("=" * 70)

display(
    skill_gap_df
)


# ------------------------------------------------------------
# 11. SUMMARY
# ------------------------------------------------------------

status_counts = (
    skill_gap_df["Status"]
    .value_counts()
)

print("\nSKILL GAP SUMMARY")
print("-" * 70)

for status in [
    "Have",
    "Partial",
    "Missing"
]:

    count = int(
        status_counts.get(status, 0)
    )

    print(
        f"{status}: {count}"
    )


# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------

if len(skill_gap_df) != len(required_skills):
    raise ValueError(
        "Skill-gap result count does not match "
        "the number of required skills."
    )

if not skill_gap_df[
    "Similarity"
].between(0, 1).all():

    raise ValueError(
        "Similarity scores must be between 0 and 1."
    )

valid_statuses = {
    "Have",
    "Partial",
    "Missing"
}

if not set(
    skill_gap_df["Status"]
).issubset(valid_statuses):

    raise ValueError(
        "Invalid skill-gap status detected."
    )


print("\n✓ Required skills embedded.")
print("✓ User skills embedded.")
print("✓ Best semantic matches calculated.")
print("✓ Skills classified as Have / Partial / Missing.")
print("✓ STEP 28 COMPLETED SUCCESSFULLY.")

STEP 28: SKILL GAP ANALYSIS
✓ Required skills found: 8
✓ Embedding model found.

USER SKILLS
----------------------------------------------------------------------
Python, Excel, SQL, Statistics

✓ User skills detected: 4

Creating skill embeddings...
✓ Required skill embeddings created.
✓ User skill embeddings created.

SKILL GAP RESULTS


,Required Skill,category,priority,Best Matching User Skill,Similarity,Status
0,SQL (Structured Query Language),Technical,High,sql,0.687,Partial
1,Python (including data‑analysis libraries such...,Technical,High,python,0.508,Partial
2,Data Visualization with Power BI and Tableau,Technical,High,excel,0.289,Missing
3,"Cloud Data‑Warehouse Platforms (AWS Redshift, ...",Technical,Medium,excel,0.122,Missing
4,Statistical Analysis with R or SAS,Technical,Medium,statistics,0.466,Missing
5,Business Requirements Documentation,Business,High,excel,0.191,Missing
6,Stakeholder Communication & Presentation (Powe...,Business,High,sql,0.071,Missing
7,Knowledge Management (maintaining model librar...,Business,Medium,sql,0.091,Missing



SKILL GAP SUMMARY
----------------------------------------------------------------------
Have: 0
Partial: 2
Missing: 6

✓ Required skills embedded.
✓ User skills embedded.
✓ Best semantic matches calculated.
✓ Skills classified as Have / Partial / Missing.
✓ STEP 28 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 29: CAREER RECOMMENDATIONS
# ============================================================

print("=" * 70)
print("STEP 29: GENERATING ADJACENT CAREER RECOMMENDATIONS")
print("=" * 70)

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. VALIDATE REQUIRED DATA
# ------------------------------------------------------------

required_vars = [
    "occupation_data",
    "required_skills_df",
    "skill_gap_df",
    "embedding_model"
]

missing_vars = [
    var for var in required_vars
    if var not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Missing required variables: {missing_vars}. "
        "Please run the previous steps first."
    )

print("✓ Occupation data found.")
print("✓ Required skills found.")
print("✓ Skill-gap analysis found.")
print("✓ Embedding model found.")


# ------------------------------------------------------------
# 2. PREPARE CURRENT JOB PROFILE
# ------------------------------------------------------------

# Get the target occupation row
target_rows = occupation_data[
    occupation_data["O*NET-SOC Code"].astype(str)
    == str(TARGET_OCCUPATION_CODE)
].copy()

if target_rows.empty:
    raise ValueError(
        f"Target occupation {TARGET_OCCUPATION_CODE} "
        "was not found in occupation_data."
    )

target_row = target_rows.iloc[0]


# Required skills
required_skill_text = ", ".join(
    required_skills_df["skill"]
    .astype(str)
    .tolist()
)

# Major tasks
task_text = " ".join(
    analysis_df["Task"]
    .astype(str)
    .head(10)
    .tolist()
)

# Build profile
target_profile = (
    f"Occupation: {target_row['Title']}. "
    f"Description: {target_row['Description']}. "
    f"Important skills: {required_skill_text}. "
    f"Major tasks: {task_text}"
)

print("\n✓ Current occupation profile created.")


# ------------------------------------------------------------
# 3. PREPARE ALL O*NET OCCUPATION PROFILES
# ------------------------------------------------------------

occupation_profiles = []

for _, row in occupation_data.iterrows():

    occupation_code = str(
        row["O*NET-SOC Code"]
    ).strip()

    title = str(
        row["Title"]
    ).strip()

    description = str(
        row["Description"]
    ).strip()

    profile = (
        f"Occupation: {title}. "
        f"Description: {description}"
    )

    occupation_profiles.append({
        "O*NET-SOC Code": occupation_code,
        "Title": title,
        "Description": description,
        "Profile": profile
    })


occupation_profiles_df = pd.DataFrame(
    occupation_profiles
)

print(
    f"✓ Occupation profiles prepared: "
    f"{len(occupation_profiles_df)}"
)


# ------------------------------------------------------------
# 4. CREATE EMBEDDING FOR CURRENT JOB
# ------------------------------------------------------------

target_embedding = embedding_model.encode(
    [target_profile],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")


# ------------------------------------------------------------
# 5. CREATE EMBEDDINGS FOR ALL OCCUPATIONS
# ------------------------------------------------------------

occupation_embeddings = embedding_model.encode(
    occupation_profiles_df["Profile"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("✓ Occupation embeddings created.")


# ------------------------------------------------------------
# 6. CALCULATE SIMILARITY
# ------------------------------------------------------------

similarities = np.dot(
    occupation_embeddings,
    target_embedding[0]
)

occupation_profiles_df[
    "Similarity"
] = similarities


# ------------------------------------------------------------
# 7. REMOVE CURRENT OCCUPATION
# ------------------------------------------------------------

career_candidates = occupation_profiles_df[
    occupation_profiles_df["O*NET-SOC Code"].astype(str)
    != str(TARGET_OCCUPATION_CODE)
].copy()


# ------------------------------------------------------------
# 8. SORT BY SIMILARITY
# ------------------------------------------------------------

career_candidates = (
    career_candidates
    .sort_values(
        by="Similarity",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. KEEP TOP 8 CANDIDATES
# ------------------------------------------------------------

career_candidates = (
    career_candidates
    .head(8)
    .copy()
)


# ------------------------------------------------------------
# 10. GENERATE SIMPLE EXPLANATIONS
# ------------------------------------------------------------

recommendation_rows = []

current_title = str(
    TARGET_OCCUPATION_TITLE
)

for _, row in career_candidates.iterrows():

    similarity = float(
        row["Similarity"]
    )

    explanation = (
        f"{row['Title']} is semantically related to "
        f"{current_title} based on occupational description "
        f"similarity ({similarity:.2f})."
    )

    recommendation_rows.append({

        "Career": row["Title"],

        "O*NET-SOC Code":
            row["O*NET-SOC Code"],

        "Similarity":
            round(similarity, 3),

        "Why It May Fit":
            explanation
    })


career_recommendations_df = pd.DataFrame(
    recommendation_rows
)


# ------------------------------------------------------------
# 11. FINAL TOP 5
# ------------------------------------------------------------

career_recommendations_df = (
    career_recommendations_df
    .head(5)
    .reset_index(drop=True)
)

career_recommendations_df[
    "Recommendation_Rank"
] = (
    career_recommendations_df.index + 1
)


# ------------------------------------------------------------
# 12. REORDER COLUMNS
# ------------------------------------------------------------

career_recommendations_df = (
    career_recommendations_df[
        [
            "Recommendation_Rank",
            "Career",
            "O*NET-SOC Code",
            "Similarity",
            "Why It May Fit"
        ]
    ]
)


# ------------------------------------------------------------
# 13. DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP ADJACENT CAREER RECOMMENDATIONS")
print("=" * 70)

display(
    career_recommendations_df
)


# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------

if career_recommendations_df.empty:
    raise ValueError(
        "No career recommendations were generated."
    )

if len(career_recommendations_df) < 3:
    raise ValueError(
        f"Expected at least 3 career recommendations, "
        f"but found {len(career_recommendations_df)}."
    )

if career_recommendations_df["Similarity"].isna().any():
    raise ValueError(
        "Career similarity contains missing values."
    )


print("\n✓ O*NET occupations compared using embeddings.")
print("✓ Current occupation excluded from recommendations.")
print(
    f"✓ {len(career_recommendations_df)} adjacent careers generated."
)
print("✓ STEP 29 COMPLETED SUCCESSFULLY.")

STEP 29: GENERATING ADJACENT CAREER RECOMMENDATIONS
✓ Occupation data found.
✓ Required skills found.
✓ Skill-gap analysis found.
✓ Embedding model found.

✓ Current occupation profile created.
✓ Occupation profiles prepared: 1016


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

✓ Occupation embeddings created.

TOP ADJACENT CAREER RECOMMENDATIONS


,Recommendation_Rank,Career,O*NET-SOC Code,Similarity,Why It May Fit
0,1,Intelligence Analysts,33-3021.06,0.690,Intelligence Analysts is semantically related ...
1,2,Data Scientists,15-2051.00,0.632,Data Scientists is semantically related to Bus...
2,3,Data Warehousing Specialists,15-1243.01,0.614,Data Warehousing Specialists is semantically r...
3,4,Database Architects,15-1243.00,0.570,Database Architects is semantically related to...
4,5,"Bookkeeping, Accounting, and Auditing Clerks",43-3031.00,0.569,"Bookkeeping, Accounting, and Auditing Clerks i..."



✓ O*NET occupations compared using embeddings.
✓ Current occupation excluded from recommendations.
✓ 5 adjacent careers generated.
✓ STEP 29 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 30: AI JOB EVOLUTION ANALYZER - FINAL AGENT
# ============================================================
#
# FEATURES
# ------------------------------------------------------------
# 1. Accepts ANY job title
# 2. Uses O*NET when a reliable occupation match exists
# 3. Uses LLM-generated tasks when no reliable O*NET match exists
# 4. Calculates:
#       - Automation Potential
#       - AI Exposure Score
#       - Human Dependency
#       - Task Classification
# 5. Identifies required skills
# 6. Accepts user's current skills
# 7. Calculates skill gaps
# 8. Recommends adjacent careers
# 9. Finds ROLE + SKILL specific learning resources
# 10. Does NOT use fixed Python/SQL-only resources
# 11. Works with roles such as:
#       plumber
#       teacher
#       educator
#       graphic designer
#       fashion designer
#       civil engineer
#       chef
#       nurse
#       accountant
#       data analyst
#       etc.
# 12. Gradio chatbot UI
#
# ============================================================

print("=" * 75)
print("STEP 30: AI JOB EVOLUTION ANALYZER - FINAL AGENT")
print("=" * 75)


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import re
import json
import glob
import html
import difflib
import urllib.parse
import warnings

import numpy as np
import pandas as pd
import requests

from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

print("✓ Core libraries imported.")


# ============================================================
# 2. INSTALL / IMPORT GRADIO
# ============================================================

try:
    import gradio as gr
    print(f"✓ Gradio version: {gr.__version__}")
except Exception:
    print("Installing Gradio...")
    !pip -q install gradio
    import gr
    print(f"✓ Gradio installed: {gr.__version__}")


# ============================================================
# 3. LOCATE O*NET JOB TITLES FILE
# ============================================================

def find_file(filename):
    """
    Search common Colab locations for a file.
    """
    candidates = [
        f"/content/{filename}",
        f"./{filename}",
        filename,
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    recursive_matches = glob.glob(
        f"/content/**/{filename}",
        recursive=True
    )

    if recursive_matches:
        return recursive_matches[0]

    return None


JOB_TITLES_PATH = find_file("job_titles.csv")

if JOB_TITLES_PATH is None:
    print("⚠ job_titles.csv was not found.")
    print("The agent will still work using generic LLM role analysis.")
    job_titles_df = pd.DataFrame()
else:
    try:
        job_titles_df = pd.read_csv(
            JOB_TITLES_PATH,
            low_memory=False
        )

        print(
            f"✓ job_titles.csv loaded: "
            f"{len(job_titles_df):,} rows"
        )

    except Exception as e:
        print("⚠ Could not load job_titles.csv:", e)
        job_titles_df = pd.DataFrame()


# ============================================================
# 4. NORMALIZE O*NET JOB TITLES DATA
# ============================================================

def normalize_text(text):
    """
    Normalize text for safer matching.
    """
    if text is None:
        return ""

    text = str(text).lower().strip()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


if not job_titles_df.empty:

    # Normalize expected columns
    for col in job_titles_df.columns:
        job_titles_df[col] = job_titles_df[col].fillna("")

    if "Job Title" in job_titles_df.columns:
        job_titles_df["__normalized_job_title"] = (
            job_titles_df["Job Title"]
            .astype(str)
            .map(normalize_text)
        )

    if "Title" in job_titles_df.columns:
        job_titles_df["__normalized_title"] = (
            job_titles_df["Title"]
            .astype(str)
            .map(normalize_text)
        )

    if "Short Title" in job_titles_df.columns:
        job_titles_df["__normalized_short_title"] = (
            job_titles_df["Short Title"]
            .astype(str)
            .map(normalize_text)
        )

    print("✓ O*NET title normalization completed.")


# ============================================================
# 5. HELPER: SAFE NUMBER
# ============================================================

def safe_float(value, default=3.0):
    try:
        value = float(value)

        if not np.isfinite(value):
            return default

        return value

    except Exception:
        return default


# ============================================================
# 6. IDENTIFY AVAILABLE GROQ CLIENT
# ============================================================

groq_client_local = None

if "groq_client" in globals():
    groq_client_local = groq_client

elif "client" in globals():
    # Only use client if it appears to be a Groq-style client
    if hasattr(client, "chat"):
        groq_client_local = client


# ============================================================
# 7. IDENTIFY AVAILABLE LLM MODEL
# ============================================================

LLM_MODEL_LOCAL = None

# Check variables already created in earlier notebook steps
for variable_name in [
    "LLM_MODEL",
    "GROQ_MODEL",
    "groq_model",
    "MODEL_NAME",
    "model_name"
]:

    if variable_name in globals():

        candidate = globals()[variable_name]

        if isinstance(candidate, str) and candidate.strip():
            LLM_MODEL_LOCAL = candidate.strip()
            break


# ============================================================
# 8. DISCOVER GROQ MODEL SAFELY
# ============================================================

if groq_client_local is not None and LLM_MODEL_LOCAL is None:

    try:

        available_models = [
            model.id
            for model in groq_client_local.models.list().data
        ]

        preferred_models = [
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b"
        ]

        for model_name_candidate in preferred_models:

            if model_name_candidate in available_models:
                LLM_MODEL_LOCAL = model_name_candidate
                break

        if LLM_MODEL_LOCAL is None and available_models:
            LLM_MODEL_LOCAL = available_models[0]

    except Exception as e:

        print(
            "⚠ Could not automatically discover Groq model:",
            e
        )


if groq_client_local is not None:
    print(
        f"✓ LLM client available | Model: "
        f"{LLM_MODEL_LOCAL}"
    )
else:
    print(
        "⚠ No LLM client detected. "
        "The agent will use deterministic fallbacks."
    )


# ============================================================
# 9. ROBUST LLM CALL
# ============================================================

def call_llm(
    prompt,
    system_message=None,
    temperature=0.2,
    max_tokens=4000
):
    """
    Safe wrapper around Groq chat completion.
    """

    if (
        groq_client_local is None
        or LLM_MODEL_LOCAL is None
    ):
        return None

    messages = []

    if system_message:
        messages.append({
            "role": "system",
            "content": system_message
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    try:

        response = groq_client_local.chat.completions.create(
            model=LLM_MODEL_LOCAL,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )

        return response.choices[0].message.content

    except Exception as e:

        print(
            "LLM call failed:",
            str(e)[:250]
        )

        return None


# ============================================================
# 10. JSON EXTRACTION HELPER
# ============================================================

def extract_json(text):
    """
    Extract the first valid JSON object/array from LLM output.
    """

    if not text:
        return None

    text = str(text).strip()

    # Remove markdown code fences
    text = re.sub(
        r"```json",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"```",
        "",
        text
    )

    text = text.strip()

    # Direct parsing
    try:
        return json.loads(text)
    except Exception:
        pass

    # Search for object
    object_match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if object_match:
        try:
            return json.loads(
                object_match.group(0)
            )
        except Exception:
            pass

    # Search for array
    array_match = re.search(
        r"\[.*\]",
        text,
        flags=re.DOTALL
    )

    if array_match:
        try:
            return json.loads(
                array_match.group(0)
            )
        except Exception:
            pass

    return None


# ============================================================
# 11. O*NET ROLE RESOLUTION
# ============================================================

def resolve_job_title(job_input):
    """
    Resolve any entered title against O*NET job_titles.csv.

    Returns:
        matched / generic
    """

    original = str(job_input).strip()
    normalized = normalize_text(original)

    if not normalized:
        return {
            "status": "invalid",
            "input_title": original,
            "onet_title": None,
            "onet_code": None,
            "match_score": 0.0
        }

    # --------------------------------------------------------
    # No O*NET file
    # --------------------------------------------------------

    if job_titles_df.empty:

        return {
            "status": "generic",
            "input_title": original,
            "onet_title": None,
            "onet_code": None,
            "match_score": 0.0
        }

    # --------------------------------------------------------
    # Exact matches
    # --------------------------------------------------------

    exact_columns = [
        "__normalized_job_title",
        "__normalized_title",
        "__normalized_short_title"
    ]

    for column in exact_columns:

        if column not in job_titles_df.columns:
            continue

        matches = job_titles_df[
            job_titles_df[column] == normalized
        ]

        if not matches.empty:

            row = matches.iloc[0]

            onet_title = (
                row["Title"]
                if "Title" in row.index
                else row.get("Job Title", original)
            )

            onet_code = (
                row["O*NET-SOC Code"]
                if "O*NET-SOC Code" in row.index
                else None
            )

            return {
                "status": "matched",
                "input_title": original,
                "onet_title": str(onet_title),
                "onet_code": (
                    str(onet_code)
                    if onet_code is not None
                    else None
                ),
                "match_score": 1.0
            }

    # --------------------------------------------------------
    # Token containment
    # --------------------------------------------------------

    normalized_title_series = None

    for column in [
        "__normalized_job_title",
        "__normalized_title",
        "__normalized_short_title"
    ]:

        if column in job_titles_df.columns:

            candidate_matches = job_titles_df[
                job_titles_df[column].str.contains(
                    re.escape(normalized),
                    case=False,
                    na=False
                )
            ]

            if not candidate_matches.empty:

                row = candidate_matches.iloc[0]

                onet_title = (
                    row["Title"]
                    if "Title" in row.index
                    else row.get("Job Title", original)
                )

                onet_code = (
                    row["O*NET-SOC Code"]
                    if "O*NET-SOC Code" in row.index
                    else None
                )

                return {
                    "status": "matched",
                    "input_title": original,
                    "onet_title": str(onet_title),
                    "onet_code": (
                        str(onet_code)
                        if onet_code is not None
                        else None
                    ),
                    "match_score": 0.90
                }

    # --------------------------------------------------------
    # Conservative fuzzy matching
    # --------------------------------------------------------

    candidate_series = None

    for column in [
        "__normalized_job_title",
        "__normalized_title",
        "__normalized_short_title"
    ]:

        if column in job_titles_df.columns:
            candidate_series = job_titles_df[column]
            break

    if candidate_series is not None:

        # Sample unique titles to reduce computation
        unique_candidates = (
            candidate_series
            .dropna()
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

        best_match = difflib.get_close_matches(
            normalized,
            unique_candidates,
            n=1,
            cutoff=0.82
        )

        if best_match:

            best_normalized = best_match[0]

            matched_rows = job_titles_df[
                candidate_series == best_normalized
            ]

            if not matched_rows.empty:

                row = matched_rows.iloc[0]

                onet_title = (
                    row["Title"]
                    if "Title" in row.index
                    else row.get(
                        "Job Title",
                        original
                    )
                )

                onet_code = (
                    row["O*NET-SOC Code"]
                    if "O*NET-SOC Code" in row.index
                    else None
                )

                similarity = difflib.SequenceMatcher(
                    None,
                    normalized,
                    best_normalized
                ).ratio()

                # Conservative threshold
                if similarity >= 0.82:

                    return {
                        "status": "matched",
                        "input_title": original,
                        "onet_title": str(onet_title),
                        "onet_code": (
                            str(onet_code)
                            if onet_code is not None
                            else None
                        ),
                        "match_score": round(
                            float(similarity),
                            3
                        )
                    }

    # --------------------------------------------------------
    # No reliable match
    # --------------------------------------------------------

    return {
        "status": "generic",
        "input_title": original,
        "onet_title": None,
        "onet_code": None,
        "match_score": 0.0
    }


# ============================================================
# 12. FETCH O*NET TASKS FOR MATCHED OCCUPATION
# ============================================================

def get_onet_tasks(onet_code):

    if not onet_code:
        return pd.DataFrame()

    # Existing notebook variable
    if "task_statements" in globals():

        try:

            df = task_statements.copy()

            code_column = "O*NET-SOC Code"

            if code_column in df.columns:

                matches = df[
                    df[code_column]
                    .astype(str)
                    == str(onet_code)
                ].copy()

                if not matches.empty:
                    return matches

        except Exception:
            pass

    # Try locating uploaded CSV
    task_path = find_file("task_statements.csv")

    if task_path:

        try:

            df = pd.read_csv(
                task_path,
                low_memory=False
            )

            matches = df[
                df["O*NET-SOC Code"]
                .astype(str)
                == str(onet_code)
            ].copy()

            return matches

        except Exception:
            pass

    return pd.DataFrame()


# ============================================================
# 13. GENERIC TASK GENERATION
# ============================================================

def generate_generic_tasks(
    role,
    job_description=""
):
    """
    Generate realistic tasks when O*NET does not provide
    a reliable match.
    """

    description_part = ""

    if job_description.strip():
        description_part = (
            f"\nJob description provided by user:\n"
            f"{job_description[:2500]}\n"
        )

    prompt = f"""
You are analyzing the occupational tasks of this job:

JOB ROLE:
{role}

{description_part}

Generate exactly 8 realistic and distinct core tasks
performed in this occupation.

The tasks must describe WORK ACTIVITIES, not skills.

Good examples:
- Install and repair water supply systems
- Inspect pipes and identify leaks
- Plan classroom lessons
- Assess student learning
- Create visual design concepts

Bad examples:
- Plumbing
- Communication
- Creativity
- Microsoft Excel

Return ONLY valid JSON using this structure:

{{
  "tasks": [
    {{
      "task_id": 1,
      "task": "..."
    }}
  ]
}}
"""

    result = call_llm(
        prompt=prompt,
        system_message=(
            "You are an occupational analysis expert. "
            "Return valid JSON only."
        ),
        temperature=0.15,
        max_tokens=2500
    )

    parsed = extract_json(result)

    if (
        isinstance(parsed, dict)
        and isinstance(parsed.get("tasks"), list)
        and len(parsed["tasks"]) >= 4
    ):

        tasks = []

        for i, item in enumerate(
            parsed["tasks"][:8],
            start=1
        ):

            task_text = str(
                item.get("task", "")
            ).strip()

            if task_text:

                tasks.append({
                    "Task ID": i,
                    "Task": task_text,
                    "Importance": 1.0,
                    "Relevance": 1.0,
                    "Frequency": 1.0,
                    "Task_Priority": 1.0
                })

        if len(tasks) >= 4:
            return pd.DataFrame(tasks)

    # --------------------------------------------------------
    # Generic deterministic fallback
    # --------------------------------------------------------

    fallback_tasks = [
        f"Perform the core day-to-day work activities required of a {role}.",
        f"Assess requirements and determine the appropriate approach for {role.lower()} work.",
        f"Use relevant tools, equipment, methods, or technology required in {role.lower()} work.",
        f"Inspect or evaluate the quality of completed {role.lower()} work.",
        f"Communicate with clients, colleagues, stakeholders, or team members.",
        f"Resolve problems and unexpected issues encountered during work.",
        f"Maintain required records, documentation, or work outputs.",
        f"Apply professional standards, safety practices, and role-specific procedures."
    ]

    rows = []

    for i, task in enumerate(
        fallback_tasks,
        start=1
    ):

        rows.append({
            "Task ID": i,
            "Task": task,
            "Importance": 1.0,
            "Relevance": 1.0,
            "Frequency": 1.0,
            "Task_Priority": 1.0
        })

    return pd.DataFrame(rows)


# ============================================================
# 14. BUILD TASK DATASET FOR ROLE
# ============================================================

def build_role_tasks(
    role,
    resolution,
    job_description=""
):

    if resolution["status"] == "matched":

        onet_tasks = get_onet_tasks(
            resolution["onet_code"]
        )

        if not onet_tasks.empty:

            # Remove duplicate task descriptions
            if "Task" in onet_tasks.columns:

                onet_tasks = (
                    onet_tasks
                    .drop_duplicates(
                        subset=["Task"]
                    )
                    .reset_index(drop=True)
                )

                # Limit to top 8 for speed
                onet_tasks = onet_tasks.head(8)

                rows = []

                for i, (_, row) in enumerate(
                    onet_tasks.iterrows(),
                    start=1
                ):

                    task_text = str(
                        row["Task"]
                    ).strip()

                    if not task_text:
                        continue

                    rows.append({
                        "Task ID": i,
                        "Task": task_text,
                        "Importance": 1.0,
                        "Relevance": 1.0,
                        "Frequency": 1.0,
                        "Task_Priority": 1.0
                    })

                if len(rows) >= 4:

                    return (
                        pd.DataFrame(rows),
                        "O*NET"
                    )

    # Generic LLM task generation
    return (
        generate_generic_tasks(
            role=role,
            job_description=job_description
        ),
        "LLM estimate"
    )


# ============================================================
# 15. RETRIEVE RAG CONTEXT WHEN AVAILABLE
# ============================================================

def get_role_rag_context(
    role,
    onet_code=None,
    top_k=5
):

    # For the exact target role already represented
    # in the notebook's RAG index
    try:

        if (
            "retrieve_rag_context" in globals()
            and "rag_index" in globals()
            and "embedding_model" in globals()
        ):

            query = role

            if onet_code:
                query += f" O*NET {onet_code}"

            retrieved = retrieve_rag_context(
                query,
                top_k=top_k
            )

            if (
                isinstance(retrieved, pd.DataFrame)
                and not retrieved.empty
            ):

                context_parts = []

                for _, row in retrieved.iterrows():

                    context_parts.append(
                        str(
                            row.get(
                                "text",
                                ""
                            )
                        )
                    )

                return "\n".join(
                    context_parts
                )[:7000]

    except Exception:
        pass

    return ""


# ============================================================
# 16. TASK-LEVEL AI SIGNALS
# ============================================================

def score_tasks_with_llm(
    role,
    tasks_df,
    rag_context=""
):

    if tasks_df.empty:
        return pd.DataFrame()

    task_payload = []

    for _, row in tasks_df.iterrows():

        task_payload.append({
            "task_id": int(
                row["Task ID"]
            ),
            "task": str(
                row["Task"]
            )
        })

    prompt = f"""
Analyze AI exposure for the occupation:

ROLE:
{role}

TASKS:
{json.dumps(task_payload, indent=2)}

Optional evidence/context:
{rag_context[:6000]}

For EVERY task provide three scores from 1 to 5:

tool_coverage:
1 = very little existing software/AI capability
5 = strong existing AI/software capability

trend_momentum:
1 = little current movement toward automation
5 = strong movement toward automation

judgment_reliance:
1 = little human judgment/context/accountability
5 = high human judgment/context/accountability/physical presence

Do not assume complete job replacement.

Return ONLY valid JSON:

{{
  "task_signals": [
    {{
      "task_id": 1,
      "tool_coverage": 1,
      "trend_momentum": 1,
      "judgment_reliance": 1,
      "rationale": "..."
    }}
  ]
}}
"""

    result = call_llm(
        prompt=prompt,
        system_message=(
            "You are an occupational AI-exposure analyst. "
            "Use conservative, task-level reasoning. "
            "Never claim that AI completely replaces an "
            "occupation. Return valid JSON only."
        ),
        temperature=0.15,
        max_tokens=5000
    )

    parsed = extract_json(result)

    signal_records = []

    if (
        isinstance(parsed, dict)
        and isinstance(
            parsed.get("task_signals"),
            list
        )
    ):

        for signal in parsed["task_signals"]:

            try:

                task_id = int(
                    signal.get("task_id")
                )

            except Exception:

                continue

            signal_records.append({
                "Task ID": task_id,
                "tool_coverage": min(
                    5,
                    max(
                        1,
                        int(
                            round(
                                safe_float(
                                    signal.get(
                                        "tool_coverage",
                                        3
                                    ),
                                    3
                                )
                            )
                        )
                    )
                ),
                "trend_momentum": min(
                    5,
                    max(
                        1,
                        int(
                            round(
                                safe_float(
                                    signal.get(
                                        "trend_momentum",
                                        3
                                    ),
                                    3
                                )
                            )
                        )
                    )
                ),
                "judgment_reliance": min(
                    5,
                    max(
                        1,
                        int(
                            round(
                                safe_float(
                                    signal.get(
                                        "judgment_reliance",
                                        3
                                    ),
                                    3
                                )
                            )
                        )
                    )
                ),
                "rationale": str(
                    signal.get(
                        "rationale",
                        "LLM task-level assessment."
                    )
                )
            })

    signals_df = pd.DataFrame(
        signal_records
    )

    # --------------------------------------------------------
    # Fallback if LLM failed
    # --------------------------------------------------------

    if signals_df.empty:

        fallback_rows = []

        for _, row in tasks_df.iterrows():

            task_text = str(
                row["Task"]
            ).lower()

            # Physical / interpersonal / high-context tasks
            physical_terms = [
                "install",
                "repair",
                "inspect",
                "operate",
                "teach",
                "care",
                "treat",
                "patient",
                "client",
                "classroom",
                "student",
                "construction",
                "plumb",
                "maintain",
                "diagnose",
                "supervise"
            ]

            digital_terms = [
                "report",
                "document",
                "analyze",
                "generate",
                "schedule",
                "calculate",
                "record",
                "data",
                "design",
                "research"
            ]

            physical_score = sum(
                term in task_text
                for term in physical_terms
            )

            digital_score = sum(
                term in task_text
                for term in digital_terms
            )

            if physical_score > digital_score:
                tool = 2
                trend = 2
                judgment = 4
            elif digital_score > physical_score:
                tool = 4
                trend = 4
                judgment = 3
            else:
                tool = 3
                trend = 3
                judgment = 3

            fallback_rows.append({
                "Task ID": int(
                    row["Task ID"]
                ),
                "tool_coverage": tool,
                "trend_momentum": trend,
                "judgment_reliance": judgment,
                "rationale": (
                    "Conservative fallback task-level "
                    "assessment."
                )
            })

        signals_df = pd.DataFrame(
            fallback_rows
        )

    return signals_df


# ============================================================
# 17. DETERMINISTIC PROJECT SCORING
# ============================================================

def calculate_exposure_scores(
    tasks_df,
    signals_df
):

    df = tasks_df.merge(
        signals_df,
        on="Task ID",
        how="left"
    )

    # Safety fallback
    for column in [
        "tool_coverage",
        "trend_momentum",
        "judgment_reliance"
    ]:

        if column not in df.columns:
            df[column] = 3

        df[column] = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .fillna(3)
            .clip(1, 5)
        )

    # Project formula
    df["Automation_Potential"] = (
        df["tool_coverage"]
    )

    df["Exposure_Score"] = (
        100
        * (
            0.5 * df["tool_coverage"]
            + 0.3 * df["trend_momentum"]
            + 0.2 * (
                6 - df["judgment_reliance"]
            )
        )
        / 5
    )

    df["Exposure_Score"] = (
        df["Exposure_Score"]
        .clip(0, 100)
        .round(2)
    )

    df["Human_Dependency"] = (
        df["judgment_reliance"]
    )

    # Classification
    def classify(row):

        automation = float(
            row["Automation_Potential"]
        )

        human = float(
            row["Human_Dependency"]
        )

        if (
            automation >= 4
            and human <= 2
        ):
            return "Highly Automatable"

        elif human >= 4:
            return "Human-dependent"

        else:
            return "AI-augmented"

    df["Classification"] = (
        df.apply(
            classify,
            axis=1
        )
    )

    # Signal status
    df["Signal_Status"] = np.where(
        df.get(
            "rationale",
            pd.Series(
                [""] * len(df)
            )
        )
        .astype(str)
        .str.contains(
            "fallback",
            case=False,
            na=False
        ),
        "Fallback",
        "LLM"
    )

    # Rank
    df = (
        df
        .sort_values(
            "Exposure_Score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    df["Exposure_Rank"] = (
        df.index + 1
    )

    # Overall score
    if (
        "Importance" in df.columns
        and df["Importance"].sum() > 0
    ):

        overall = (
            (
                df["Exposure_Score"]
                * df["Importance"]
            ).sum()
            / df["Importance"].sum()
        )

    else:

        overall = df["Exposure_Score"].mean()

    return df, round(
        float(overall),
        2
    )


# ============================================================
# 18. SKILL EXTRACTION + GAP ANALYSIS
# ============================================================

def generate_skill_analysis(
    role,
    task_df,
    user_skills
):

    task_text = "\n".join(
        f"- {row['Task']}"
        for _, row in task_df.iterrows()
    )

    user_skills_text = (
        user_skills
        if user_skills.strip()
        else "No skills provided."
    )

    prompt = f"""
Analyze skills for this occupation:

ROLE:
{role}

CORE TASKS:
{task_text[:9000]}

USER'S CURRENT SKILLS:
{user_skills_text}

Return ONLY valid JSON.

Rules:
1. Identify 8 important role-specific skills.
2. These must be relevant to the actual occupation.
3. Do NOT force technology skills into non-technology jobs.
4. A plumber should receive plumbing-related skills.
5. A teacher should receive teaching-related skills.
6. A designer should receive design-related skills.
7. For each skill classify the user's level as:
   Have
   Partial
   Missing
8. Provide a short reason.
9. Missing/Partial skills become learning targets.

Format:

{{
  "skills": [
    {{
      "skill": "...",
      "status": "Have",
      "reason": "..."
    }}
  ]
}}
"""

    result = call_llm(
        prompt=prompt,
        system_message=(
            "You are a career-skills analyst. "
            "Return valid JSON only."
        ),
        temperature=0.15,
        max_tokens=4500
    )

    parsed = extract_json(result)

    if (
        isinstance(parsed, dict)
        and isinstance(
            parsed.get("skills"),
            list
        )
    ):

        records = []

        allowed_statuses = {
            "have": "Have",
            "partial": "Partial",
            "missing": "Missing"
        }

        for item in parsed["skills"][:8]:

            skill = str(
                item.get(
                    "skill",
                    ""
                )
            ).strip()

            if not skill:
                continue

            raw_status = str(
                item.get(
                    "status",
                    "Missing"
                )
            ).strip().lower()

            status = allowed_statuses.get(
                raw_status,
                "Missing"
            )

            records.append({
                "Skill": skill,
                "Status": status,
                "Reason": str(
                    item.get(
                        "reason",
                        ""
                    )
                ).strip()
            })

        if records:
            return pd.DataFrame(records)

    # --------------------------------------------------------
    # Deterministic fallback
    # --------------------------------------------------------

    # Use LLM-less generic skills only as emergency fallback
    fallback_skill_map = {
        "plumber": [
            "Plumbing installation",
            "Pipe repair",
            "Leak detection",
            "Fixture installation",
            "Plumbing codes",
            "Safety procedures",
            "Blueprint reading",
            "Troubleshooting"
        ],
        "teacher": [
            "Lesson planning",
            "Classroom management",
            "Student assessment",
            "Instructional methods",
            "Communication",
            "Curriculum development",
            "Educational technology",
            "Inclusive teaching"
        ],
        "educator": [
            "Lesson planning",
            "Classroom management",
            "Student assessment",
            "Instructional methods",
            "Communication",
            "Curriculum development",
            "Educational technology",
            "Inclusive teaching"
        ]
    }

    role_key = normalize_text(role)

    selected = None

    for key, skills in fallback_skill_map.items():

        if key in role_key:
            selected = skills
            break

    if selected is None:

        selected = [
            f"Core {role} knowledge",
            f"{role} tools and techniques",
            "Problem solving",
            "Professional communication",
            "Quality assurance",
            "Safety and professional standards",
            "Documentation",
            "Industry-specific practices"
        ]

    # Parse user skills
    known_user_skills = [
        normalize_text(skill)
        for skill in re.split(
            r"[,;\n]",
            user_skills
        )
        if normalize_text(skill)
    ]

    records = []

    for skill in selected:

        normalized_skill = normalize_text(
            skill
        )

        if any(
            normalized_skill in user_skill
            or user_skill in normalized_skill
            for user_skill in known_user_skills
        ):

            status = "Have"

        else:

            status = "Missing"

        records.append({
            "Skill": skill,
            "Status": status,
            "Reason": (
                "Fallback role-specific skill analysis."
            )
        })

    return pd.DataFrame(records)


# ============================================================
# 19. CAREER RECOMMENDATIONS
# ============================================================

def generate_career_recommendations(
    role,
    skills_df
):

    skills = [
        str(skill)
        for skill in skills_df["Skill"].tolist()
    ]

    prompt = f"""
Suggest 5 occupations adjacent to:

ROLE:
{role}

Relevant skills:
{json.dumps(skills)}

Return ONLY valid JSON:

{{
  "careers": [
    {{
      "title": "...",
      "reason": "..."
    }}
  ]
}}

Rules:
- Recommend realistic related occupations.
- Do not invent absurd career transitions.
- Keep explanations concise.
"""

    result = call_llm(
        prompt=prompt,
        system_message=(
            "You are a career transition analyst. "
            "Return valid JSON only."
        ),
        temperature=0.2,
        max_tokens=2500
    )

    parsed = extract_json(result)

    if (
        isinstance(parsed, dict)
        and isinstance(
            parsed.get("careers"),
            list
        )
    ):

        careers = []

        for item in parsed["careers"][:5]:

            title = str(
                item.get(
                    "title",
                    ""
                )
            ).strip()

            if not title:
                continue

            careers.append({
                "Career": title,
                "Reason": str(
                    item.get(
                        "reason",
                        ""
                    )
                ).strip()
            })

        if careers:
            return pd.DataFrame(
                careers
            )

    return pd.DataFrame([
        {
            "Career": f"Senior {role}",
            "Reason": (
                "Natural progression within the same occupation."
            )
        },
        {
            "Career": f"Specialist {role}",
            "Reason": (
                "Build deeper domain specialization."
            )
        },
        {
            "Career": f"Supervisor - {role}",
            "Reason": (
                "Develop leadership and coordination skills."
            )
        }
    ])


# ============================================================
# 20. WEB SEARCH FUNCTION
# ============================================================
#
# IMPORTANT:
# This is the part that makes learning resources dynamic.
#
# It searches using:
#
#     ROLE + MISSING SKILL + course/training
#
# Therefore:
#
# plumber + pipe repair
# teacher + classroom management
# designer + typography
#
# instead of hard-coded tech skills.
# ============================================================

def web_search_resources(
    query,
    max_results=5
):
    """
    Lightweight web search using DuckDuckGo HTML.

    No search API key is required.
    If external web access is unavailable, the function
    returns an empty list and the fallback search links
    are used.
    """

    encoded_query = urllib.parse.quote_plus(
        query
    )

    url = (
        "https://html.duckduckgo.com/html/?q="
        + encoded_query
    )

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/120 Safari/537.36"
        )
    }

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=8
        )

        if response.status_code != 200:
            return []

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        results = []

        for result in soup.select(
            ".result"
        )[:max_results]:

            title_node = result.select_one(
                ".result__a"
            )

            snippet_node = result.select_one(
                ".result__snippet"
            )

            if title_node is None:
                continue

            title = (
                title_node.get_text(
                    " ",
                    strip=True
                )
            )

            href = title_node.get(
                "href",
                ""
            )

            snippet = ""

            if snippet_node is not None:

                snippet = (
                    snippet_node.get_text(
                        " ",
                        strip=True
                    )
                )

            # Ignore malformed links
            if not href:
                continue

            results.append({
                "title": title,
                "url": href,
                "snippet": snippet
            })

        return results

    except Exception:

        return []


# ============================================================
# 21. SAFE FALLBACK SEARCH LINKS
# ============================================================

def build_fallback_learning_links(
    role,
    skill
):
    """
    Create direct search URLs on major learning platforms.

    These are fallback links only.
    They do NOT pretend to be specific courses when we
    could not verify a live course page.
    """

    query = urllib.parse.quote_plus(
        f"{role} {skill}"
    )

    coursera_url = (
        "https://www.coursera.org/search?query="
        + query
    )

    udemy_url = (
        "https://www.udemy.com/courses/search/?q="
        + query
    )

    edx_url = (
        "https://www.edx.org/search?q="
        + query
    )

    youtube_url = (
        "https://www.youtube.com/results?search_query="
        + query
    )

    google_url = (
        "https://www.google.com/search?q="
        + urllib.parse.quote_plus(
            f"{role} {skill} course training"
        )
    )

    return [
        {
            "title": f"Coursera — {skill}",
            "url": coursera_url,
            "provider": "Coursera"
        },
        {
            "title": f"Udemy — {skill}",
            "url": udemy_url,
            "provider": "Udemy"
        },
        {
            "title": f"edX — {skill}",
            "url": edx_url,
            "provider": "edX"
        },
        {
            "title": f"YouTube Learning — {skill}",
            "url": youtube_url,
            "provider": "YouTube"
        },
        {
            "title": f"Web search — {role} {skill}",
            "url": google_url,
            "provider": "Google"
        }
    ]


# ============================================================
# 22. DYNAMIC LEARNING RESOURCE RECOMMENDATION
# ============================================================

def get_learning_resources(
    role,
    skills_df,
    max_skills=4
):

    if skills_df.empty:
        return []

    target_rows = skills_df[
        skills_df["Status"].isin(
            ["Missing", "Partial"]
        )
    ]

    if target_rows.empty:
        return []

    target_rows = target_rows.head(
        max_skills
    )

    resources = []

    for _, row in target_rows.iterrows():

        skill = str(
            row["Skill"]
        ).strip()

        status = str(
            row["Status"]
        ).strip()

        if not skill:
            continue

        # ----------------------------------------------------
        # DYNAMIC SEARCH QUERY
        # ----------------------------------------------------

        query = (
            f"{role} {skill} "
            f"course training learning"
        )

        live_results = web_search_resources(
            query,
            max_results=4
        )

        skill_resources = []

        # ----------------------------------------------------
        # Use live results where possible
        # ----------------------------------------------------

        for result in live_results:

            url = str(
                result.get(
                    "url",
                    ""
                )
            ).strip()

            title = str(
                result.get(
                    "title",
                    ""
                )
            ).strip()

            if not url or not title:
                continue

            skill_resources.append({
                "skill": skill,
                "status": status,
                "title": title,
                "url": url,
                "provider": "Web search",
                "description": str(
                    result.get(
                        "snippet",
                        ""
                    )
                ).strip()
            })

        # ----------------------------------------------------
        # Always provide fallback links when necessary
        # ----------------------------------------------------

        if len(skill_resources) < 2:

            fallback_links = (
                build_fallback_learning_links(
                    role,
                    skill
                )
            )

            existing_urls = {
                r["url"]
                for r in skill_resources
            }

            for item in fallback_links:

                if item["url"] in existing_urls:
                    continue

                skill_resources.append({
                    "skill": skill,
                    "status": status,
                    "title": item["title"],
                    "url": item["url"],
                    "provider": item["provider"],
                    "description": (
                        f"Search for {skill} "
                        f"learning resources relevant "
                        f"to {role}."
                    )
                })

                if len(skill_resources) >= 4:
                    break

        resources.extend(
            skill_resources[:4]
        )

    return resources


# ============================================================
# 23. HTML ESCAPE
# ============================================================

def esc(value):
    return html.escape(
        str(value)
    )


# ============================================================
# 24. FORMAT ANALYSIS REPORT
# ============================================================

def create_report(
    role,
    resolution,
    task_source,
    analysis_df,
    overall_score,
    skills_df,
    careers_df,
    resources,
    user_skills
):

    matched_title = (
        resolution.get(
            "onet_title"
        )
        or role
    )

    onet_code = resolution.get(
        "onet_code"
    )

    if resolution["status"] == "matched":

        evidence_label = (
            f"O*NET-supported analysis "
            f"(matched: {matched_title})"
        )

        if onet_code:
            evidence_label += (
                f" — {onet_code}"
            )

    else:

        evidence_label = (
            "LLM occupational estimate "
            "because no reliable O*NET title match "
            "was found."
        )

    # --------------------------------------------------------
    # Overall interpretation
    # --------------------------------------------------------

    if overall_score >= 70:
        interpretation = (
            "Higher task-level AI exposure. "
            "Several activities have substantial "
            "current or emerging automation capability."
        )

    elif overall_score >= 50:
        interpretation = (
            "Moderate task-level AI exposure. "
            "AI is likely to transform some activities "
            "while human involvement remains important."
        )

    else:
        interpretation = (
            "Lower task-level AI exposure. "
            "Many activities continue to rely strongly "
            "on human judgment, physical work, or context."
        )

    # --------------------------------------------------------
    # Task table
    # --------------------------------------------------------

    task_table = []

    for _, row in analysis_df.iterrows():

        short_task = str(
            row["Task"]
        )

        if len(short_task) > 115:
            short_task = (
                short_task[:112]
                + "..."
            )

        task_table.append(
            f"""
<tr>
<td>{int(row["Exposure_Rank"])}</td>
<td>{esc(short_task)}</td>
<td>{row["Exposure_Score"]:.1f}</td>
<td>{esc(row["Classification"])}</td>
</tr>
"""
        )

    task_table_html = "".join(
        task_table
    )

    # --------------------------------------------------------
    # Skills table
    # --------------------------------------------------------

    skill_rows = []

    for _, row in skills_df.iterrows():

        status = row["Status"]

        skill_rows.append(
            f"""
<tr>
<td>{esc(row["Skill"])}</td>
<td><b>{esc(status)}</b></td>
<td>{esc(row["Reason"])}</td>
</tr>
"""
        )

    skills_html = "".join(
        skill_rows
    )

    # --------------------------------------------------------
    # Career recommendations
    # --------------------------------------------------------

    career_rows = []

    for _, row in careers_df.iterrows():

        career_rows.append(
            f"""
<tr>
<td><b>{esc(row["Career"])}</b></td>
<td>{esc(row["Reason"])}</td>
</tr>
"""
        )

    careers_html = "".join(
        career_rows
    )

    # --------------------------------------------------------
    # Learning resources
    # --------------------------------------------------------

    resources_html = ""

    if resources:

        grouped_resources = {}

        for resource in resources:

            grouped_resources.setdefault(
                (
                    resource["skill"],
                    resource["status"]
                ),
                []
            )

            grouped_resources[
                (
                    resource["skill"],
                    resource["status"]
                )
            ].append(resource)

        for (
            skill,
            status
        ), items in grouped_resources.items():

            resources_html += (
                f"<h4>{esc(skill)} "
                f"<small>({esc(status)})</small></h4>"
            )

            resources_html += "<ul>"

            for item in items:

                resources_html += (
                    "<li>"
                    f'<a href="{esc(item["url"])}" '
                    'target="_blank">'
                    f'{esc(item["title"])}'
                    "</a> "
                    f"<small>"
                    f"— {esc(item['provider'])}"
                    "</small>"
                    f"<br>"
                    f"<span>"
                    f"{esc(item['description'])}"
                    f"</span>"
                    "</li>"
                )

            resources_html += "</ul>"

    else:

        resources_html = (
            "<p>No skill gaps were identified, "
            "so additional learning resources "
            "are not required.</p>"
        )

    # --------------------------------------------------------
    # MAIN REPORT
    # --------------------------------------------------------

    report = f"""
<div style="
    font-family: Arial, sans-serif;
    line-height: 1.55;
    max-width: 1100px;
">

<h1>🤖 AI Job Evolution Analyzer</h1>

<h2>{esc(role.title())}</h2>

<p>
<b>Analysis basis:</b> {esc(evidence_label)}
</p>

<hr>

<h2>📊 Job-Level AI Exposure</h2>

<div style="
    font-size: 34px;
    font-weight: bold;
    margin: 10px 0;
">
{overall_score}/100
</div>

<p>
{esc(interpretation)}
</p>

<p>
<b>Important:</b>
This score represents task-level exposure and
potential transformation. It is not a prediction
that the entire occupation will be replaced by AI.
</p>

<hr>

<h2>🔍 Task Analysis</h2>

<table border="1"
cellpadding="8"
cellspacing="0"
style="border-collapse: collapse; width:100%;">
<thead>
<tr>
<th>Rank</th>
<th>Task</th>
<th>AI Exposure</th>
<th>Classification</th>
</tr>
</thead>

<tbody>
{task_table_html}
</tbody>
</table>

<p>
<b>Task source:</b> {esc(task_source)}
</p>

<hr>

<h2>🧠 Skill Analysis</h2>

<p>
<b>Your current skills:</b>
{esc(user_skills if user_skills.strip() else "Not provided")}
</p>

<table border="1"
cellpadding="8"
cellspacing="0"
style="border-collapse: collapse; width:100%;">
<thead>
<tr>
<th>Skill</th>
<th>Status</th>
<th>Reason</th>
</tr>
</thead>

<tbody>
{skills_html}
</tbody>
</table>

<hr>

<h2>🎓 Skill-Gap Learning Resources</h2>

<p>
Resources are selected according to the
<b>actual role + missing/partial skill</b>.
They are not limited to technology jobs.
</p>

{resources_html}

<hr>

<h2>🚀 Related Career Paths</h2>

<table border="1"
cellpadding="8"
cellspacing="0"
style="border-collapse: collapse; width:100%;">
<thead>
<tr>
<th>Career</th>
<th>Why it is related</th>
</tr>
</thead>

<tbody>
{careers_html}
</tbody>
</table>

<hr>

<h3>ℹ️ Analysis Information</h3>

<p>
<b>Role entered:</b> {esc(role)}
</p>

<p>
<b>O*NET match:</b>
{esc(matched_title)}
</p>

<p>
<b>O*NET code:</b>
{esc(onet_code or "Not available")}
</p>

<p>
<b>Task classification:</b>
Highly Automatable / AI-augmented / Human-dependent
</p>

</div>
"""

    return report


# ============================================================
# 25. MAIN AGENT FUNCTION
# ============================================================

def analyze_job(
    job_role,
    job_description="",
    current_skills=""
):

    job_role = str(
        job_role or ""
    ).strip()

    job_description = str(
        job_description or ""
    ).strip()

    current_skills = str(
        current_skills or ""
    ).strip()

    if not job_role:

        return (
            "Please enter a job role.",
            [],
            "No analysis performed."
        )

    # --------------------------------------------------------
    # Resolve role
    # --------------------------------------------------------

    resolution = resolve_job_title(
        job_role
    )

    # --------------------------------------------------------
    # Build task set
    # --------------------------------------------------------

    tasks_df, task_source = (
        build_role_tasks(
            role=job_role,
            resolution=resolution,
            job_description=job_description
        )
    )

    if tasks_df.empty:

        return (
            "Unable to generate tasks for this role.",
            [],
            "Task generation failed."
        )

    # --------------------------------------------------------
    # RAG context
    # --------------------------------------------------------

    rag_context = get_role_rag_context(
        role=(
            resolution.get(
                "onet_title"
            )
            or job_role
        ),
        onet_code=resolution.get(
            "onet_code"
        ),
        top_k=5
    )

    # --------------------------------------------------------
    # LLM task signals
    # --------------------------------------------------------

    signals_df = score_tasks_with_llm(
        role=job_role,
        tasks_df=tasks_df,
        rag_context=rag_context
    )

    # --------------------------------------------------------
    # Deterministic scoring
    # --------------------------------------------------------

    analysis_df, overall_score = (
        calculate_exposure_scores(
            tasks_df,
            signals_df
        )
    )

    # --------------------------------------------------------
    # Skill analysis
    # --------------------------------------------------------

    skills_df = generate_skill_analysis(
        role=job_role,
        task_df=analysis_df,
        user_skills=current_skills
    )

    # --------------------------------------------------------
    # Related careers
    # --------------------------------------------------------

    careers_df = generate_career_recommendations(
        role=job_role,
        skills_df=skills_df
    )

    # --------------------------------------------------------
    # DYNAMIC LEARNING RESOURCES
    # --------------------------------------------------------

    resources = get_learning_resources(
        role=job_role,
        skills_df=skills_df,
        max_skills=4
    )

    # --------------------------------------------------------
    # Build final report
    # --------------------------------------------------------

    report = create_report(
        role=job_role,
        resolution=resolution,
        task_source=task_source,
        analysis_df=analysis_df,
        overall_score=overall_score,
        skills_df=skills_df,
        careers_df=careers_df,
        resources=resources,
        user_skills=current_skills
    )

    # --------------------------------------------------------
    # Simple agent metadata
    # --------------------------------------------------------

    metadata = {
        "role": job_role,
        "match_status": resolution["status"],
        "onet_title": resolution.get(
            "onet_title"
        ),
        "onet_code": resolution.get(
            "onet_code"
        ),
        "task_source": task_source,
        "overall_score": overall_score,
        "tasks_analyzed": len(
            analysis_df
        ),
        "skills_analyzed": len(
            skills_df
        ),
        "learning_resources_found": len(
            resources
        )
    }

    metadata_text = json.dumps(
        metadata,
        indent=2
    )

    return (
        report,
        analysis_df,
        metadata_text
    )


# ============================================================
# 26. PRE-FLIGHT TESTS
# ============================================================

print("\n" + "=" * 75)
print("PRE-FLIGHT TEST")
print("=" * 75)

test_roles = [
    "Business Intelligence Analyst",
    "Plumber",
    "Teacher",
    "Graphic Designer"
]

for test_role in test_roles:

    result = resolve_job_title(
        test_role
    )

    matched_title = (
        result.get(
            "onet_title",
            "NO RELIABLE MATCH"
        )
    )

    status = result.get(
        "status",
        "unknown"
    )

    print(
        f"{test_role:<35} -> "
        f"{matched_title} "
        f"[{status}]"
    )

print("✓ Pre-flight role resolution completed.")


# ============================================================
# 27. GRADIO AGENT UI
# ============================================================

print("\n" + "=" * 75)
print("BUILDING GRADIO AGENT")
print("=" * 75)


# Store conversation
chat_history = []


def agent_submit(
    job_role,
    job_description,
    current_skills,
    history
):

    history = history or []

    # Analyze
    report, analysis_df, metadata_text = (
        analyze_job(
            job_role=job_role,
            job_description=job_description,
            current_skills=current_skills
        )
    )

    user_message = (
        f"Analyze the job role: {job_role}"
    )

    if job_description.strip():

        user_message += (
            "\n\nJob description:\n"
            + job_description
        )

    if current_skills.strip():

        user_message += (
            "\n\nMy current skills:\n"
            + current_skills
        )

    # IMPORTANT:
    # Gradio Chatbot here expects message dictionaries.
    history = list(history)

    history.append({
        "role": "user",
        "content": user_message
    })

    history.append({
        "role": "assistant",
        "content": report
    })

    return (
        history,
        "",
        "",
        "",
        metadata_text
    )


def clear_agent():

    return (
        [],
        "",
        "",
        "",
        ""
    )


with gr.Blocks(
    title="AI Job Evolution Analyzer"
) as demo:

    gr.Markdown(
        """
# 🤖 AI Job Evolution Analyzer

### Assess AI exposure • Identify skill gaps • Discover learning resources • Explore career paths

Enter **any job role** — not just technology roles.

Examples:

`Plumber`

`Teacher`

`Graphic Designer`

`Civil Engineer`

`Chef`

`Data Analyst`

`Nurse`
        """
    )

    with gr.Row():

        with gr.Column(
            scale=1
        ):

            job_role_input = gr.Textbox(
                label="Job Role",
                placeholder=(
                    "e.g. Plumber, Teacher, "
                    "Graphic Designer, Data Analyst"
                ),
                lines=1
            )

            job_description_input = gr.Textbox(
                label="Job Description (Optional)",
                placeholder=(
                    "Paste a job description here "
                    "for a more specific analysis..."
                ),
                lines=5
            )

            skills_input = gr.Textbox(
                label="Your Current Skills (Optional)",
                placeholder=(
                    "e.g. communication, CAD, "
                    "teaching, plumbing..."
                ),
                lines=3
            )

            with gr.Row():

                analyze_button = gr.Button(
                    "🔍 Analyze Job",
                    variant="primary"
                )

                clear_button = gr.Button(
                    "🗑 Clear"
                )

            metadata_output = gr.Textbox(
                label="Agent Metadata",
                lines=10
            )

        with gr.Column(
            scale=2
        ):

            chatbot = gr.Chatbot(
                label="AI Agent",
                height=650
            )


    # --------------------------------------------------------
    # Button action
    # --------------------------------------------------------

    analyze_button.click(
        fn=agent_submit,
        inputs=[
            job_role_input,
            job_description_input,
            skills_input,
            chatbot
        ],
        outputs=[
            chatbot,
            job_role_input,
            job_description_input,
            skills_input,
            metadata_output
        ]
    )

    # --------------------------------------------------------
    # Enter key action
    # --------------------------------------------------------

    job_role_input.submit(
        fn=agent_submit,
        inputs=[
            job_role_input,
            job_description_input,
            skills_input,
            chatbot
        ],
        outputs=[
            chatbot,
            job_role_input,
            job_description_input,
            skills_input,
            metadata_output
        ]
    )

    # --------------------------------------------------------
    # Clear
    # --------------------------------------------------------

    clear_button.click(
        fn=clear_agent,
        inputs=[],
        outputs=[
            chatbot,
            job_role_input,
            job_description_input,
            skills_input,
            metadata_output
        ]
    )


# ============================================================
# 28. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("STEP 30 READY")
print("=" * 75)

print("✓ Any job title can be entered.")
print("✓ O*NET matching enabled.")
print("✓ Generic LLM role analysis enabled.")
print("✓ Task-level AI exposure analysis enabled.")
print("✓ Automation Potential calculated.")
print("✓ Human Dependency calculated.")
print("✓ Task Classification calculated.")
print("✓ Role-specific skills generated.")
print("✓ User skill-gap analysis enabled.")
print("✓ Related career recommendations enabled.")
print("✓ DYNAMIC role + skill learning-resource search enabled.")
print("✓ Fallback learning links enabled.")
print("✓ Gradio chatbot uses message dictionaries.")
print("✓ No Gradio 'type' argument used.")
print("✓ No hard-coded technology-only learning dictionary.")
print("=" * 75)


# ============================================================
# 29. LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=False
)

STEP 30: AI JOB EVOLUTION ANALYZER - FINAL AGENT
✓ Core libraries imported.
✓ Gradio version: 6.26.0
⚠ job_titles.csv was not found.
The agent will still work using generic LLM role analysis.
⚠ No LLM client detected. The agent will use deterministic fallbacks.

PRE-FLIGHT TEST
Business Intelligence Analyst       -> None [generic]
Plumber                             -> None [generic]
Teacher                             -> None [generic]
Graphic Designer                    -> None [generic]
✓ Pre-flight role resolution completed.

BUILDING GRADIO AGENT

STEP 30 READY
✓ Any job title can be entered.
✓ O*NET matching enabled.
✓ Generic LLM role analysis enabled.
✓ Task-level AI exposure analysis enabled.
✓ Automation Potential calculated.
✓ Human Dependency calculated.
✓ Task Classification calculated.
✓ Role-specific skills generated.
✓ User skill-gap analysis enabled.
✓ Related career recommendations enabled.
✓ DYNAMIC role + skill learning-resource search enabled.
✓ Fallback learning l